# 74 · What continuous analysis along the PT adds to S1/S2/S3, on the package pseudospace

**The question.** On the package pseudospace, what does a continuous analysis along the proximal tubule (PT) show
that the S1/S2/S3 segment analysis does not, and which of those additions survive step-only nulls, a placebo
coordinate and the coordinate's own positional noise?

Notebooks 57–61 answered this on SCF13 with Gaussian gene models on log1p. Here every gene model is the NB model of
notebooks 69/71 (raw counts, offset log of the mt-free library, notebook 71's fixed per-gene dispersion), and the
units, labels and coordinate are the package's (notebook 70). Every Gaussian statistic gets its NB counterpart,
defined in the frozen protocol before any fit: `docs/results/pt-package-continuous-vs-discrete.md`.

| Result | Old notebook | What it asks |
|---|---|---|
| R1 | 57 | Is the signal beyond S1/S2/S3 steps within-segment biology, or label–coordinate discordance? |
| R2 | 61, 37 | What does the continuous screen call that the segment screen (notebook 75) does not? Ladder rows |
| R3 | 61 | Worked examples: one gene three ways (whole PT, S1/S2/S3, curve) |
| R4 | 58 | Per-specimen descriptors: onset order, within-segment slopes, interior extrema, boundary shifts |
| R5 | 59 | Do onsets and slopes replicate in external mouse nuclei? |
| R6 | 60 | Resolution budget, the placebo coordinate and WSR; the claim gates |
| R7 | new | Solute-transporter micro-gradients (Figure 3b) |

**Inputs.** Notebook 70's export and run folders; notebook 71's stage cache and tables (α, curves, calls, universe,
gene sets); notebook 72's refit payloads and tables (noise budget, C14, registration); notebook 75's segment
F-test, DESeq2 rows and programs; Census mouse PT nuclei; HGNC gene groups 752 and 417.

**Outputs** (`results/pt_package_continuous_vs_discrete/`): `tables/`, `figures/`, `stage_cache/`, `refits/placebo/`
and `run_record.json`.

**Order.** Sections that need neither notebook 72's tables nor notebook 75's run first. Sections that read them are
guarded (G3, G4): when those tables are absent they are listed as pending and the notebook is rerun once they land
(stage-cache hits make the rerun cheap).

Cohort: 2 male control mice and 2 cortex sections of 1 male human donor. The two human sections are not donor
replicates. Everything here is descriptive within this cohort.

In [ ]:
import os

# Resource caps (notebook 74 shares the machine): about 24 threads; at most one package refit at a time.
for _name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMBA_NUM_THREADS'):
    os.environ.setdefault(_name, '24')

import argparse
import json
import re
import subprocess
import sys
import time
from hashlib import sha256
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
parser.add_argument('--visium-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser().resolve()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser().resolve()
visium_root = args.visium_root or os.environ.get('PSEUDOSPACE_VISIUM_ROOT')

NOTEBOOK_LOGIC_VERSION = '74.package.1'   # Bump when a cached calculation changes.
EXTERNAL_VERSION, EXTERNAL_COMMIT, HARMONY_VERSION = '0.1.0', '59da04b', '2.0.5'
CONFIRMED_PASS1 = {'hvm': 'efafc9ac2e99', 'aki': 'bb4a80c1b484'}   # coordinator's label checkpoint (notebook 70)
DS_NAME = {'hvm': 'human_vs_mouse', 'aki': 'aki_vs_control'}
SEGMENTS = ('PT-S1', 'PT-S2', 'PT-S3')
SHORT_SEG = {'PT-S1': 'S1', 'PT-S2': 'S2', 'PT-S3': 'S3'}
MICE, HUMANS = ('Ctrl1A2', 'Ctrl1A4'), ('HUK1_COR1', 'HUK1_MED1')
PAIRS = {'mouse': MICE, 'human': HUMANS}
BASIS_DF, GRID_POINTS = 6, 101                     # notebook 71's gene model
ALPHA, CAMERA_COR, CALIBRATION_MAX = .05, .01, .075  # notebook 71's set test and pass rule
N_REPLICATES = 10                                  # per step-null arm
ARM_SEEDS_57, NULL_SEEDS_58 = {'A': 5701, 'B': 5801}, {'A': 5901, 'B': 6001}
DECISION_HIGH, DECISION_LOW = .5, .2               # R1 (57's rule)
MIN_AMPLITUDE, MIN_ABS_RHO, MIN_RANGE = .5, .9, .1  # R4 (58's rules), read on the natural-log scale (NB note)
MIN_PROMINENCE, INNER = .2, .2
SLOPE_CLAIM, SLOPE_SPLIT = .3, .2
CLAIM_PRIMARY, CLAIM_SPLIT, ABANDON = .3, .2, .1
SHIFT_GENES, SHIFT_PROGRAMS, PROGRAM_MIN_GENES = 10, 3, 3
N_BOOT, BOOT_SEED = 2000, 58
MIN_SLOPE_GENES = 50                               # 58: a cell's slope agreement needs > 50 genes
SPLIT_EPSILON, SPLIT_SEED = .5, 35                 # notebook 35 and 72's count split
N_LADDER_BINS = 6                                  # 37's six coordinate bins
EXAMPLE_BINS, REVERSAL, NEAR_MISS = 12, .25, .25   # R3 and Fig 3e
PROBE_EFFECT, PROBE_Q = .7, .10                    # 37's equal-probe rule
T_MIN_CALLS, LOCAL_SLOPE, T_MIN_SLOPE_GENES = 10, 2., 20   # R7
WSR_PERMUTATIONS, WSR_SEED, WSR_ALPHA, WSR_DF = 500, 61, .01, 4   # notebook 53's WSR
NOISE_DOMINATED, C14_CELLS = 1., 6                 # 72's C14
HGNC = {752: ('Solute carriers', '131e82911a0840e98f70c6c3075059d0a69a8709c0a3c43db69029aed3a97bee'),
        417: ('ATP binding cassette transporters', 'cc6cdac305cd75465de8ae34faac429a0356db916171bfeba4ec8d4ce82cf28a')}
HGNC_DOWNLOADED = '2026-10-12'
RUN_COMPARATORS = True                             # O1: the M section is separable; False drops it and nothing else
N_JOBS = int(os.environ.get('PSEUDOSPACE_N_JOBS', 24))   # never part of a cache key
PLACEBO_THREADS = 24                               # the placebo refit's BLAS/OpenMP threads (resource cap)

package_out = results_root / 'package_pseudospace'
RUN = {ds: package_out / 'runs' / ds for ds in DS_NAME}
EXPORT = {ds: package_out / ds / 'pt_units.h5ad' for ds in DS_NAME}
CONFIG_HVM = project / 'analysis' / 'package_configs' / 'human_vs_mouse'
nb71 = results_root / 'pt_pathways_package_pseudospace'
nb72 = results_root / 'pt_package_coordinate_validation'
nb75 = results_root / 'pt_package_pathway_methods'
output = results_root / 'pt_package_continuous_vs_discrete'
tables, figures, cache = output / 'tables', output / 'figures', output / 'stage_cache'
REFIT_DIR = output / 'refits'
for folder in (tables, figures, cache, REFIT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
REQUIRED = [*EXPORT.values(), results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv',
            nb71 / 'tables' / 'universe_human_vs_mouse.csv', nb71 / 'tables' / 'gene_sets_human_vs_mouse.json',
            nb71 / 'tables' / 'universe_aki_vs_control.csv', nb71 / 'tables' / 'gene_sets_aki_vs_control.json',
            nb71 / 'tables' / 'pathway_tests_human_vs_mouse_cameraPR_species.csv',
            nb71 / 'tables' / 'redundancy_clusters_human_vs_mouse.csv',
            data_root / 'external' / 'census_pt_cells' / 'mouse_pt_cells.h5ad',
            results_root / 'pt_literature_deep' / 'probe_counts.csv',
            *[data_root / 'external' / 'hgnc' / f'hgnc_group_{g}.tsv' for g in HGNC]]
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')
READING, PENDING = [], []   # how the protocol was read where it is silent; sections waiting for notebooks 72 or 75
print('Results folder:', output, '· workers:', N_JOBS)

## 1 · Guards (not cached)

- The package is 0.1.0 at 59da04b; R harmony is 2.0.5 (the placebo refit runs the package).
- Notebook 70's pass-1 fingerprints equal the confirmed ones (human vs mouse `efafc9ac2e99`, AKI `bb4a80c1b484`), in
  the run manifests and in the export provenance.
- The 15 genes that set the package's labels or orientation (PT panels and axis anchors) are read from the tracked
  configuration.

In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from importlib.metadata import distribution, version
from IPython.display import display
from scipy import sparse
from scipy.stats import false_discovery_control, norm, spearmanr

pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 120)
direct_url = json.loads(distribution('pseudospace-reconstruction').read_text('direct_url.json') or '{}')
EXTERNAL = {'version': version('pseudospace-reconstruction'),
            'commit': direct_url.get('vcs_info', {}).get('commit_id', 'unknown')}
assert EXTERNAL['version'] == EXTERNAL_VERSION and EXTERNAL['commit'].startswith(EXTERNAL_COMMIT), EXTERNAL
harmony = subprocess.run([sys.executable, '-c', (
    'import os, sys; os.environ.setdefault("R_HOME", os.path.join(sys.prefix, "lib", "R")); '
    'import rpy2.robjects as ro; print(ro.r("as.character(packageVersion(\'harmony\'))")[0])')],
    capture_output=True, text=True).stdout.strip().splitlines()
assert harmony and harmony[-1] == HARMONY_VERSION, f'R harmony must be {HARMONY_VERSION}: {harmony}'

from pseudospace import continuous_descriptors as descriptors_module
from pseudospace import discrete_comparators, nb_gam, pathway_calibration, pathway_pipelines, within_segment_nb
from pseudospace import package_refits as refit_module
from pseudospace.continuous_descriptors import (blockwise_permutation, bootstrap_partial_spearman, coordinate_segments,
                                                interior_extrema, logistic_transitions, monotone_onsets,
                                                partial_spearman, segment_units, switch_fraction)
from pseudospace.coordinate_evaluation import log_normalize, poisson_count_split, sha256_folds
from pseudospace.discrete_comparators import swap_rule
from pseudospace.nb_gam import nb_dispersion, nested_nb_lr
from pseudospace.package_inputs import aki_inputs, hvm_inputs, read_pt_units, support_positions
from pseudospace.pathway_calibration import contrast_designs
from pseudospace.pathway_pipelines import camera_pr, common_grid, moderated_f, partition_groups
from pseudospace.stage_cache import cached_payload, code_digest, digest, stage_key
from pseudospace.stats_gam import gam_internal_knots, make_gam_design
from pseudospace.within_segment_nb import (beyond_step_designs, nb_segment_slopes, nb_specimen_curves, nested_lr,
                                           pseudobulk_log_rates, step_ladder_designs, within_species_designs)

UNITS = {}
for ds in DS_NAME:
    run_fp = json.loads((RUN[ds] / '03_integration' / 'manifest.json').read_text())['summary']['fingerprint_pass1']['membership_sha1']
    UNITS[ds] = read_pt_units(EXPORT[ds], ds)
    export_fp = UNITS[ds].uns['provenance']['fingerprints']['pass1']['membership_sha1']
    assert run_fp == export_fp == CONFIRMED_PASS1[ds], (ds, run_fp, export_fp, CONFIRMED_PASS1[ds])
PANELS = yaml.safe_load((CONFIG_HVM / 'marker_panels.yaml').read_text())
FIXED_GENES = sorted({g for s in SEGMENTS for g in PANELS['label_panels'][s]}
                     | set(PANELS['axis_anchors']['early']) | set(PANELS['axis_anchors']['late']))
assert len(FIXED_GENES) == 15, FIXED_GENES
CODE = digest([NOTEBOOK_LOGIC_VERSION] + [Path(m.__file__) for m in (nb_gam, pathway_calibration, pathway_pipelines,
                                                                     within_segment_nb, descriptors_module)])
SECONDS = {}


def cached(stage, compute, *, params=None, inputs=None):
    """cached_payload under this notebook's root and code; keeps the stage's uncached compute time."""
    def wrapped():
        t = time.perf_counter()
        out = dict(compute())
        out['__seconds'] = np.array(time.perf_counter() - t)
        return out
    result = cached_payload(stage, wrapped, root=cache, params=params, inputs=inputs, code=CODE)
    SECONDS[stage] = float(result.pop('__seconds'))
    return result


def stored(folder, stage):
    """Another notebook's newest cached payload of one stage (read only), or None when it is absent."""
    files = sorted((folder / 'stage_cache').glob(f'{stage}__*.npz'),
                   key=lambda p: json.loads(Path(str(p) + '.key.json').read_text())['written_utc'])
    if not files:
        return None
    with np.load(files[-1], allow_pickle=False) as payload:
        return {k: payload[k] for k in payload.files}


def slug(label):
    return label.replace(':', '_').replace('+', '_')


def bh(p):
    return false_discovery_control(np.asarray(p, float))


def set_z(p_one_sided):
    """Set-level z from cameraPR's one-sided p (upper tail)."""
    return norm.isf(np.clip(np.asarray(p_one_sided, float), 1e-300, 1))


print('Guards passed:', EXTERNAL, '· R harmony', harmony[-1], '· confirmed pass-1 fingerprints', CONFIRMED_PASS1)
print('Genes that set labels or orientation:', ', '.join(FIXED_GENES))

## 2 · Inputs: notebook 71's universe, gene sets and gene model

`package_inputs.hvm_inputs` with notebook 71's call (the 13 `mt-` genes out of the library and the universe). The
universe and gene sets are notebook 71's in every analysis here.

**Guards.**
- G1: the builder reproduces notebook 71: the counts digest in 71's cache key, 11,055 genes (`universe_*.csv`) and
  1,513 sets (`gene_sets_*.json`); AKI likewise (for the ladder rows only).
- G2: the NB LR of the group × spline term, refitted here with 71's α, reproduces 71's cached condition T_spatial
  (relative difference < 10⁻⁴; the nested fits warm-start differently).

In [ ]:
orthologs = pd.read_csv(results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv')
MT = sorted(g for g in orthologs.mouse_symbol.astype(str) if g.startswith('mt-'))
assert len(MT) == 13, MT
kept_h = hvm_inputs(UNITS['hvm'], data_root, orthologs)                      # mt kept: notebook 71's AKI pool
HM = hvm_inputs(UNITS['hvm'], data_root, orthologs, exclude_genes=MT)
GENES = list(HM['genes'])
SETS = HM['gene_sets']
assert GENES == pd.read_csv(nb71 / 'tables' / 'universe_human_vs_mouse.csv', index_col=0).index.astype(str).tolist()
assert SETS == json.loads((nb71 / 'tables' / 'gene_sets_human_vs_mouse.json').read_text())
key71 = sorted((nb71 / 'stage_cache').glob('gene_model_human_vs_mouse_species__*.npz.key.json'))[-1]
counts71 = re.search(r"'counts': '([0-9a-f]{16})'", json.loads(key71.read_text())['inputs']).group(1)
assert digest(HM['counts'].tocsr()) == counts71, 'notebook 71 analysed other counts'
M71 = stored(nb71, 'gene_model_human_vs_mouse_species')
ALPHA71 = {'alpha': M71['alpha'], 'converged': M71['alpha_converged'].astype(bool)}

U = UNITS['hvm']
IDS = pd.Index(U.obs_names.astype(str))
SPEC_ALL = U.obs['sample'].astype(str).to_numpy()
SEG_ALL = U.obs.segment_class.astype(str).to_numpy()
HUMAN_ALL = U.obs.group.astype(str).eq('Human').to_numpy()
RAW_ALL = U.obs.pseudospace.to_numpy(float)
DPT_ALL = U.obs.dpt_pseudotime.to_numpy(float)
EXPORT_GENES = pd.Index(U.var_names.astype(str))
X_ALL = sparse.csr_matrix(U.X)
MT_FREE = ~EXPORT_GENES.isin(MT)
GENE_COLS = EXPORT_GENES.get_indexer(GENES)
COUNTS_ALL = X_ALL[:, GENE_COLS].tocsr()
LIB_ALL = np.asarray(X_ALL[:, MT_FREE].sum(axis=1)).ravel().astype(float)


def view(raw, name, *, alpha=None, counts=None, library=None, segment=None):
    """One coordinate's analysis set: the units with a finite position inside every specimen's [1st, 99th] common
    support, positions min-max scaled over them (notebook 71's rule), 71's genes, and the mt-free library."""
    raw = np.asarray(raw, float)
    finite = np.flatnonzero(np.isfinite(raw))
    seg_all = SEG_ALL if segment is None else np.asarray(segment).astype(str)
    mask, position = support_positions(raw[finite], SPEC_ALL[finite], seg_all[finite])
    units = finite[mask]
    c = (COUNTS_ALL if counts is None else counts)[units].tocsr()
    lib = (LIB_ALL if library is None else library)[units]
    v = {'name': name, 'units': units, 'ids': IDS[units], 'position': position, 'specimen': SPEC_ALL[units],
         'segment': seg_all[units], 'human': HUMAN_ALL[units], 'counts': c, 'library': lib, 'alpha': alpha,
         'knots': gam_internal_knots(position, basis_df=BASIS_DF),
         'grid': common_grid(position, SPEC_ALL[units], GRID_POINTS),
         'log_mean_count': np.log(np.maximum(np.asarray(c.mean(axis=0)).ravel(), 1e-12))}
    v['coord_segment'], v['thresholds'] = coordinate_segments(position, v['segment'], v['specimen'])
    v['partitions'] = partition_groups(v['specimen'], v['human'])
    v['key'] = {'counts': digest(c), 'library': lib, 'position': position, 'specimen': v['specimen'],
                'segment': v['segment'], 'genes': GENES}
    return v


V = {'P': view(RAW_ALL, 'P', alpha=ALPHA71)}
P = V['P']
assert list(P['ids']) == list(HM['obs'].index.astype(str))
np.testing.assert_allclose(P['position'], HM['position'], rtol=0, atol=1e-12)
assert (P['counts'] != HM['counts'].tocsr()).nnz == 0 and np.allclose(P['library'], HM['library'])
np.testing.assert_allclose(P['grid'], M71['grid'], rtol=0, atol=1e-12)
np.testing.assert_allclose(P['knots'], M71['knots'], rtol=0, atol=1e-12)
LABELS = list(P['partitions'])
SHORT = dict(zip(LABELS, ['condition', 'relabeling 1', 'relabeling 2']))
assert LABELS[0] == 'species'

designs_g2, weights_g2 = contrast_designs(P['position'], P['human'].astype(float), P['specimen'], P['knots'])
g2 = cached('g2_t_spatial', lambda: {k: v for k, v in nested_nb_lr(
    P['counts'], designs_g2, weights_g2, np.log(P['library']), ALPHA71, {'T_spatial': ('level', 'full')},
    n_jobs=N_JOBS).items() if k in ('T_spatial', 'converged')},
            params={'basis_df': BASIS_DF}, inputs={**P['key'], 'alpha': ALPHA71['alpha']})
rel = np.abs(g2['T_spatial'] - M71['stat|T_spatial']) / np.maximum(1, np.abs(M71['stat|T_spatial']))
assert rel.max() < 1e-4, rel.max()
del designs_g2, weights_g2

# AKI (ladder rows only)
AK = aki_inputs(UNITS['aki'], data_root, orthologs, kept_h['genes'], exclude_genes=MT)
assert list(AK['genes']) == pd.read_csv(nb71 / 'tables' / 'universe_aki_vs_control.csv', index_col=0).index.astype(str).tolist()
assert AK['gene_sets'] == json.loads((nb71 / 'tables' / 'gene_sets_aki_vs_control.json').read_text())
key71a = sorted((nb71 / 'stage_cache').glob('gene_model_aki_vs_control_species__*.npz.key.json'))[-1]
assert digest(AK['counts'].tocsr()) == re.search(r"'counts': '([0-9a-f]{16})'", json.loads(key71a.read_text())['inputs']).group(1)
MA71 = stored(nb71, 'gene_model_aki_vs_control_species')
A_COUNTS = AK['counts'].tocsr()
AKI = {'name': 'AKI', 'position': np.asarray(AK['position'], float), 'specimen': AK['specimen'].astype(str),
       'segment': np.asarray(AK['segment']).astype(str), 'human': np.asarray(AK['case'], bool), 'counts': A_COUNTS,
       'library': np.asarray(AK['library'], float),
       'alpha': {'alpha': MA71['alpha'], 'converged': MA71['alpha_converged'].astype(bool)},
       'log_mean_count': np.log(np.maximum(np.asarray(A_COUNTS.mean(axis=0)).ravel(), 1e-12)),
       'genes': list(AK['genes']), 'sets': AK['gene_sets']}
AKI['knots'] = gam_internal_knots(AKI['position'], basis_df=BASIS_DF)
AKI['partitions'] = partition_groups(AKI['specimen'], AKI['human'])
AKI['key'] = {'counts': digest(A_COUNTS), 'library': AKI['library'], 'position': AKI['position'],
              'specimen': AKI['specimen'], 'segment': AKI['segment'], 'genes': AKI['genes']}
np.testing.assert_allclose(AKI['knots'], MA71['knots'], rtol=0, atol=1e-12)
del kept_h, AK

CALLS71 = pd.read_csv(nb71 / 'tables' / 'pathway_tests_human_vs_mouse_cameraPR_species.csv').set_index('pathway_id')
C71 = set(CALLS71.index[CALLS71.call.astype(bool)])
STAT71 = pd.read_csv(nb71 / 'tables' / 'gene_statistics_human_vs_mouse.csv', header=[0, 1], index_col=0)['condition']
CLUSTERS = {ds: pd.read_csv(nb71 / 'tables' / f'redundancy_clusters_{DS_NAME[ds]}.csv').set_index('pathway_id').cluster_id
            for ds in DS_NAME}
print(f"P: {len(P['ids']):,} units · {len(GENES):,} genes · {len(SETS):,} sets · 71 calls {len(C71)} · "
      f"G2 max relative difference {rel.max():.2e}")
print(f"AKI: {len(AKI['position']):,} units · {len(AKI['genes']):,} genes · {len(AKI['sets']):,} sets")

## 3 · Coordinates and fold protection

| Code | Coordinate | α |
|---|---|---|
| P | `pseudospace` on 71's scale (primary) | 71's |
| D | the package DPT (`dpt_pseudotime`, finite units; sensitivity, D3) | its own, under D's species-split M_full |
| E | notebook 72's equal-depth refit, full counts (the depth leg of the extrema rule) | 71's (a refit) |
| H | notebook 72's half-A refit, half-B counts (the count-split arm) | 71's (binomial thinning keeps NB α) |

A refit that does not run S1 → S3 in every specimen is failed (72's rule) and its arm is not evaluated.

**G5.** Half B is regenerated with 72's own call (`poisson_count_split` on the base `units_qc` counts, ε 0.5,
`default_rng(35)`), and checked by reproducing the stage key of 72's G6 arm b, whose inputs hold 72's half-B counts.

**Fold protection.** Genes that built the coordinate are excluded from per-gene descriptors: the base run's
integration features (pass 1 ∪ pass 2 `highly_variable_for_harmony`) and the 15 panel and anchor genes. Package limit
(72's O4): the labels still use all genes.

In [ ]:
BASE_FEATURES = set()
for name in ('pass1_labelled.h5ad', 'pass2.h5ad'):
    obj = ad.read_h5ad(RUN['hvm'] / '03_integration' / name, backed='r')
    BASE_FEATURES |= set(obj.var_names[obj.var['highly_variable_for_harmony'].to_numpy(bool)].astype(str))
    obj.file.close()
FOLD_PROTECTED = {'P': ~pd.Index(GENES).isin(sorted(BASE_FEATURES | set(FIXED_GENES)))}
for code in ('D', 'E', 'H'):
    FOLD_PROTECTED[code] = FOLD_PROTECTED['P']

alpha_d_view = view(DPT_ALL, 'D')
dd, dw = contrast_designs(alpha_d_view['position'], alpha_d_view['human'].astype(float), alpha_d_view['specimen'],
                          alpha_d_view['knots'])
alpha_d = cached('alpha_D', lambda: {k: v for k, v in nb_dispersion(alpha_d_view['counts'], dd['full'], dw,
                                                                     np.log(alpha_d_view['library']), n_jobs=N_JOBS).items()
                                     if k in ('alpha', 'converged')},
                 params={'basis_df': BASIS_DF}, inputs=alpha_d_view['key'])
alpha_d_view['alpha'] = {'alpha': alpha_d['alpha'], 'converged': alpha_d['converged'].astype(bool)}
V['D'] = alpha_d_view
del dd, dw

REFITS72 = {name: stored(nb72, f'refit_hvm_{name}') for name in ('fold0', 'fold1', 'fold2', 'half_A', 'equal_depth')}
REFIT_ROWS = []


def refit_coordinate(name, column='pseudospace'):
    """A 72 refit's coordinate on every export unit (NaN where it dropped a unit), or None if absent or failed."""
    p = REFITS72[name]
    if p is None:
        return None
    z = refit_module.refit_position(p, IDS, column)
    if column == 'dpt_pseudotime':
        z, _ = refit_module.orient_like_export(z, SEG_ALL)
    ok = refit_module.runs_s1_to_s3(z, SPEC_ALL, SEG_ALL)
    REFIT_ROWS.append({'refit': name, 'column': column, 'units with a position': int(np.isfinite(z).sum()),
                       'runs S1 → S3 in every specimen': ok})
    return z if ok else None


z_ed = refit_coordinate('equal_depth')
if z_ed is not None:
    V['E'] = view(z_ed, 'E', alpha=ALPHA71)
else:
    PENDING.append('E: notebook 72\'s equal-depth refit is absent or failed')

base_qc = ad.read_h5ad(RUN['hvm'] / '02_qc' / 'units_qc.h5ad')
assert pd.Index(base_qc.var_names.astype(str)).equals(EXPORT_GENES)
PT_ROWS = base_qc.obs_names.get_indexer(IDS)
assert (PT_ROWS >= 0).all()
BASE_COUNTS = sparse.csr_matrix(base_qc.layers['counts']).astype(np.int64)
assert np.array_equal(np.asarray(BASE_COUNTS[PT_ROWS].sum(axis=1)).ravel(), np.asarray(X_ALL.sum(axis=1)).ravel())
_, HALF_B_ALL = poisson_count_split(BASE_COUNTS, SPLIT_EPSILON, np.random.default_rng(SPLIT_SEED))
HALF_B = HALF_B_ALL[PT_ROWS]
del HALF_B_ALL
g6_key = sorted((nb72 / 'stage_cache').glob('g6_b__*.npz'))
U71_72 = pd.read_csv(nb71 / 'tables' / 'universe_human_vs_mouse.csv')
U71_72 = U71_72.loc[U71_72.eligible.astype(bool), 'gene'].astype(str).tolist() if 'eligible' in U71_72 else GENES
expected_key = stage_key('g6_b', params={'arm': 'b · base coordinate', 'basis_df': 6, 'df1': 6, 'camera': .01},
                         inputs={'half_b': HALF_B, 'raw': RAW_ALL, 'universe': U71_72,
                                 'sets': json.loads((nb71 / 'tables' / 'gene_sets_human_vs_mouse.json').read_text())},
                         code=code_digest(digest(['72.package_coordinate_validation.1'])))
G5 = bool(g6_key) and g6_key[-1].name == f'g6_b__{expected_key}.npz'
if not G5:
    READING.append('G5: the stage key of 72\'s G6 arm b was not reproduced; half B is checked only by construction '
                   '(72\'s call, seed and base counts).')
z_half_a = refit_coordinate('half_A')
if z_half_a is not None:
    HALF_B_COUNTS = HALF_B[:, GENE_COLS].astype(float).tocsr()
    HALF_B_LIB = np.asarray(HALF_B[:, MT_FREE].sum(axis=1)).ravel().astype(float)
    keep_h = np.where(HALF_B_LIB > 0, z_half_a, np.nan)
    V['H'] = view(keep_h, 'H', alpha=ALPHA71, counts=HALF_B_COUNTS, library=HALF_B_LIB)
    SPLIT_CONSTRUCTION = set(REFITS72['half_A']['pass1_features'].astype(str)) | set(REFITS72['half_A']['pass2_features'].astype(str))
else:
    SPLIT_CONSTRUCTION = set()
    PENDING.append('H: notebook 72\'s half-A refit is absent or failed')
del base_qc
COORD_TABLE = pd.DataFrame([{'coordinate': code, 'units': len(v['ids']), 'α': 'own' if code == 'D' else "71's",
                             'counts': 'half B' if code == 'H' else 'full',
                             **{f'units {n}': int((v['specimen'] == n).sum()) for n in (*MICE, *HUMANS)},
                             'label agreement of coordinate segments (mean)': float(v['thresholds']['label agreement'].mean())}
                            for code, v in V.items()])
COORD_TABLE.to_csv(tables / 'coordinates.csv', index=False)
display(COORD_TABLE.round(3))
display(pd.DataFrame(REFIT_ROWS))
print('G5 (72\'s half-B counts reproduced through its G6 stage key):', G5)
print(f'Fold-protected genes: {int(FOLD_PROTECTED["P"].sum()):,} of {len(GENES):,} '
      f'({len(BASE_FEATURES):,} base integration features, 15 panel and anchor genes)')

## 4 · R1 · Is the signal beyond S1/S2/S3 steps within-segment biology? (notebook 57, NB)

- **Test A** (beyond **label** steps): numerator LR(shared spline + shared L-steps + group × L-steps → + group ×
  spline), df₁ 6; denominator LR(that model + specimen × L-steps → + specimen × spline), df₂ 12 (relabeling 6).
- **Test B**: the same with coordinate-threshold segments C (per specimen, the two cuts that best reproduce its
  labels).
- Relabelings carry the species shape as nuisance in the test's own basis (species × spline and species × steps).
- Shared steps enter every model (protocol, O8). Notebook 57's own design (steps for the case group only) runs
  beside it on P as a sensitivity, with the same mirrored denominator.
- Gene z from `moderated_f`; sets by cameraPR (0.01); a call is Up with FDR ≤ 0.05; 71's pass rule.
- **Semi-synthetic arms** (species split only): arm A permutes count rows with their library inside specimen × C
  (truth = coordinate steps; labels noisy), arm B inside specimen × L (truth = label steps; the coordinate noisy).
  10 replicates each, seeds 5701+ and 5801+, α fixed, every numerator and denominator refitted.
- **Decision** (57's): arm-A median test-A calls ÷ observed test-A calls ≥ 0.5 → label–coordinate discordance;
  < 0.2 → the label-noise explanation is rejected; otherwise partly explained. The floor (arm B test A, arm A test B,
  which should call nothing) is reported.
- **Within-segment candidate:** called by A and B on both P and D; **strict:** also both set z on P above every
  simulated set z of the same test.

In [ ]:
def ftest(lr, covariate):
    """Notebook 71's moderated F on an LR pair: z, p and gene q (BH over the universe)."""
    frame = moderated_f(pd.Series(lr['numerator'], index=GENES), int(lr['df1']), np.asarray(lr['denominator'], float),
                        int(lr['df2']), covariate=covariate, robust=False)
    frame['q'] = bh(frame.p.to_numpy())
    frame['converged'] = np.asarray(lr['converged'], bool)
    frame['separated'] = np.asarray(lr['separated'], bool)
    return frame


def camera(z, sets=None):
    return camera_pr(pd.Series(np.asarray(z, float), index=GENES if sets is None else z.index),
                     SETS if sets is None else sets, inter_gene_cor=CAMERA_COR, use_ranks=False, fdr=ALPHA)


def lr_payload(lr):
    return {k: np.asarray(lr[k]) for k in ('numerator', 'denominator', 'df1', 'df2', 'converged', 'separated')}


def r1_lr(v, label, test, *, counts=None, library=None, shared=True):
    group, nuisance = v['partitions'][label]
    segments = v['segment'] if test == 'A' else v['coord_segment']
    designs, comparisons, weights = beyond_step_designs(v['position'], group, v['specimen'], v['knots'], segments,
                                                        nuisance=nuisance, shared_steps=shared)
    return lr_payload(nested_lr(v['counts'] if counts is None else counts, v['library'] if library is None else library,
                                designs, comparisons, weights, v['alpha'], n_jobs=N_JOBS))


def pass_row(tests, label_calls=None):
    """71's pass rule on {label: cameraPR table} of one statistic."""
    calls = {label: int(t.call.sum()) for label, t in tests.items()}
    relabeled = [calls[label] for label in LABELS[1:]]
    pooled = np.concatenate([tests[label].p_one_sided.to_numpy() for label in LABELS[1:]])
    return swap_rule(calls['species'], relabeled, len(tests['species']), pooled)


R1_GENES, R1_SETS = {}, {}
TESTS_R1 = {'A': 'test A (beyond label steps)', 'B': 'test B (beyond coordinate steps)'}
for code in ('P', 'D'):
    v = V[code]
    alpha_key = v['alpha']['alpha']
    variants = [('A', True), ('B', True)] + ([('A', False), ('B', False)] if code == 'P' else [])
    for test, shared in variants:
        name = test if shared else test + '57'
        for label in LABELS:
            lr = cached(f'r1_{code}_{name}_{slug(label)}', lambda v=v, label=label, test=test, shared=shared:
                        r1_lr(v, label, test, shared=shared),
                        params={'test': test, 'shared_steps': shared, 'label': label, 'basis_df': BASIS_DF},
                        inputs={**v['key'], 'alpha': alpha_key, 'coord_segment': v['coord_segment']})
            assert (int(lr['df1']), int(lr['df2'])) == (6, 12 if label == 'species' else 6), \
                (code, name, label, int(lr['df1']), int(lr['df2']))
            frame = ftest(lr, v['log_mean_count'])
            R1_GENES[(code, name, label)] = frame
            R1_SETS[(code, name, label)] = camera(frame.z)
for code in ('P', 'D'):
    pd.concat({f'{name} · {SHORT[label]}': R1_GENES[(c, name, label)][['F', 'z', 'q', 'converged', 'separated']]
               for (c, name, label) in R1_GENES if c == code}, axis=1).to_csv(tables / f'beyond_steps_genes_{code}.csv')
    pd.concat({f'{name} · {SHORT[label]}': R1_SETS[(c, name, label)].set_index('pathway_id')
               for (c, name, label) in R1_SETS if c == code}, axis=1).to_csv(tables / f'beyond_steps_sets_{code}.csv')


def arm_replicate(v, arm, rep):
    blocks = np.array([f'{a}|{b}' for a, b in zip(v['specimen'], v['coord_segment'] if arm == 'A' else v['segment'])])
    index = blockwise_permutation(blocks, np.random.default_rng(ARM_SEEDS_57[arm] + rep))
    counts, library = v['counts'][index], v['library'][index]
    out = {}
    for test in 'AB':
        frame = ftest(r1_lr(v, 'species', test, counts=counts, library=library), v['log_mean_count'])
        t = camera(frame.z)
        out[f'{test}|p_one_sided'] = t.p_one_sided.to_numpy()
        out[f'{test}|call'] = t.call.to_numpy(bool)
        out[f'{test}|share_p05'] = np.array((frame.p <= .05).mean())
    return out


ARMS_R1 = {}
for code in ('P', 'D'):
    v = V[code]
    for arm in 'AB':
        for rep in range(1, N_REPLICATES + 1):
            ARMS_R1[(code, arm, rep)] = cached(
                f'r1_arm_{code}_{arm}_{rep}', lambda v=v, arm=arm, rep=rep: arm_replicate(v, arm, rep),
                params={'arm': arm, 'rep': rep, 'seed': ARM_SEEDS_57[arm], 'basis_df': BASIS_DF},
                inputs={**v['key'], 'alpha': v['alpha']['alpha'], 'coord_segment': v['coord_segment']})

DECISION_ROWS, PASS_ROWS = [], []
for code in ('P', 'D'):
    names = ['A', 'B'] + (['A57', 'B57'] if code == 'P' else [])
    for name in names:
        tests = {label: R1_SETS[(code, name, label)] for label in LABELS}
        PASS_ROWS.append({'coordinate': code, 'statistic': name, **pass_row(tests),
                          'relabeled calls': ', '.join(str(int(tests[label].call.sum())) for label in LABELS[1:])})
    obs = {name: int(R1_SETS[(code, name, 'species')].call.sum()) for name in names}
    arm_calls = pd.DataFrame([{'arm': arm, 'rep': rep, 'test': test, 'calls': int(ARMS_R1[(code, arm, rep)][f'{test}|call'].sum()),
                               'share p ≤ 0.05': float(ARMS_R1[(code, arm, rep)][f'{test}|share_p05'])}
                              for arm in 'AB' for rep in range(1, N_REPLICATES + 1) for test in 'AB'])
    arm_calls.assign(coordinate=code).to_csv(tables / f'beyond_steps_arms_{code}.csv', index=False)
    med = arm_calls.groupby(['arm', 'test']).calls.median()
    rng_ = arm_calls.groupby(['arm', 'test']).calls.agg(['min', 'max'])
    ratio_a = med[('A', 'A')] / max(obs['A'], 1)
    ratio_b = med[('B', 'B')] / max(obs['B'], 1)
    decision = ('label–coordinate discordance (no within-segment claim from test A)' if ratio_a >= DECISION_HIGH else
                'label-noise explanation rejected' if ratio_a < DECISION_LOW else 'partly explained by label noise')
    DECISION_ROWS.append({
        'coordinate': code, 'test A calls (beyond label steps)': obs['A'], 'test B calls (beyond coordinate steps)': obs['B'],
        'relabeled test A': ', '.join(str(int(R1_SETS[(code, 'A', l)].call.sum())) for l in LABELS[1:]),
        'relabeled test B': ', '.join(str(int(R1_SETS[(code, 'B', l)].call.sum())) for l in LABELS[1:]),
        'arm A median test A (range)': f"{med[('A', 'A')]:g} ({rng_.loc[('A', 'A'), 'min']}–{rng_.loc[('A', 'A'), 'max']})",
        'arm A median test B': med[('A', 'B')], 'arm B median test A': med[('B', 'A')],
        'arm B median test B': med[('B', 'B')], 'primary ratio (arm A test A ÷ observed A)': ratio_a,
        'arm B test B ÷ observed B': ratio_b, 'floor: arm B test A, arm A test B (medians)': f"{med[('B', 'A')]:g}, {med[('A', 'B')]:g}",
        "57's design: test A, test B calls": f"{obs['A57']}, {obs['B57']}" if code == 'P' else 'not run',
        'label agreement mouse / human': ' / '.join(
            f"{V[code]['thresholds'][V[code]['thresholds'].specimen.isin(PAIRS[sp])]['label agreement'].mean():.3f}"
            for sp in ('mouse', 'human')),
        'decision': decision})
R1_DECISION = pd.DataFrame(DECISION_ROWS).set_index('coordinate')
R1_PASS = pd.DataFrame(PASS_ROWS)
R1_DECISION.to_csv(tables / 'beyond_steps_decision.csv')
R1_PASS.to_csv(tables / 'beyond_steps_pass_rules.csv', index=False)
pd.concat([V[c]['thresholds'].assign(coordinate=c) for c in V]).to_csv(tables / 'coordinate_thresholds.csv', index=False)
display(R1_DECISION.T)
display(R1_PASS.round(3))

# Per-pathway: observed set z, simulated maxima, candidates
per = {}
for code in ('P', 'D'):
    for test in 'AB':
        t = R1_SETS[(code, test, 'species')].set_index('pathway_id')
        sim = np.max([set_z(ARMS_R1[(code, arm, rep)][f'{test}|p_one_sided']) for arm in 'AB'
                      for rep in range(1, N_REPLICATES + 1)], axis=0)
        per[f'{code} {test} called'] = t.call.astype(bool)
        per[f'{code} {test} set z'] = pd.Series(set_z(t.p_one_sided), index=t.index)
        per[f'{code} {test} max simulated set z'] = pd.Series(sim, index=t.index)
CAND = pd.DataFrame(per)
CAND['candidate on P'] = CAND['P A called'] & CAND['P B called']
CAND['candidate on D'] = CAND['D A called'] & CAND['D B called']
CAND['within-segment candidate (P and D)'] = CAND['candidate on P'] & CAND['candidate on D']
CAND['strict (P set z above every simulated)'] = (CAND['within-segment candidate (P and D)']
                                                  & (CAND['P A set z'] > CAND['P A max simulated set z'])
                                                  & (CAND['P B set z'] > CAND['P B max simulated set z']))
CAND['called by 71'] = CAND.index.isin(sorted(C71))
CAND.insert(0, 'name', CAND.index.str.split('::').str[1].str.replace(r' R-HSA-\d+$', '', regex=True))
CAND.to_csv(tables / 'within_segment_candidates.csv')
print('Within-segment candidates (A and B on P and D):', int(CAND['within-segment candidate (P and D)'].sum()),
      '· strict:', int(CAND['strict (P set z above every simulated)'].sum()),
      '· of them called by 71:', int((CAND['within-segment candidate (P and D)'] & CAND['called by 71']).sum()))

## 5 · R2 (part 1) · The ladder rows: steps on the spline and six coordinate bins (37's rows, NB)

Group × steps beyond a group offset, with the shared spline and shared steps in every model, judged against
specimen × steps (the mirrored denominator):
- **steps on the spline:** the package labels; df₁ 2, df₂ 4 (relabeling 2);
- **six coordinate bins:** six equal-width bins of the analysed position; df₁ 5, df₂ 10 (relabeling 5).

Both datasets (75's O5): human vs mouse on P, AKI on notebook 71's AKI inputs with 71's AKI α. Saved in notebook 75's
layout (`pathway_tests_ladder_{row}_{ds}.csv`, column blocks per partition) so 75's method table can merge them.

In [ ]:
def ladder_lr(v, label, segments):
    group, nuisance = v['partitions'][label]
    designs, comparisons, weights = step_ladder_designs(v['position'], group, v['specimen'], v['knots'], segments,
                                                        nuisance=nuisance)
    return lr_payload(nested_lr(v['counts'], v['library'], designs, comparisons, weights, v['alpha'], n_jobs=N_JOBS))


LADDER_TESTS, LADDER_ROWS = {}, []
EXPECTED_DF = {'steps_on_spline': (2, 4, 2), 'six_bins': (5, 10, 5)}
for ds, v in (('hvm', P), ('aki', AKI)):
    genes, sets = (GENES, SETS) if ds == 'hvm' else (v['genes'], v['sets'])
    labels = list(v['partitions'])
    short = dict(zip(labels, ['condition', 'relabeling 1', 'relabeling 2']))
    bins = np.clip((v['position'] * N_LADDER_BINS).astype(int), 0, N_LADDER_BINS - 1).astype(str)
    for row, segments in (('steps_on_spline', v['segment']), ('six_bins', bins)):
        tests = {}
        for label in labels:
            lr = cached(f'ladder_{ds}_{row}_{slug(label)}', lambda v=v, label=label, segments=segments:
                        ladder_lr(v, label, segments), params={'row': row, 'label': label, 'basis_df': BASIS_DF},
                        inputs={**v['key'], 'alpha': v['alpha']['alpha'], 'segments': segments})
            df1, df2c, df2r = EXPECTED_DF[row]
            assert (int(lr['df1']), int(lr['df2'])) == (df1, df2c if label == labels[0] else df2r), (ds, row, label)
            frame = moderated_f(pd.Series(lr['numerator'], index=genes), int(lr['df1']), np.asarray(lr['denominator'], float),
                                int(lr['df2']), covariate=v['log_mean_count'], robust=False)
            tests[label] = camera_pr(frame.z, sets, inter_gene_cor=CAMERA_COR, use_ranks=False, fdr=ALPHA)
        LADDER_TESTS[(ds, row)] = tests
        pd.concat({short[label]: t.set_index('pathway_id') for label, t in tests.items()}, axis=1).to_csv(
            tables / f'pathway_tests_ladder_{row}_{DS_NAME[ds]}.csv')
        calls = {label: set(t.pathway_id[t.call.astype(bool)]) for label, t in tests.items()}
        rule = swap_rule(len(calls[labels[0]]), [len(calls[l]) for l in labels[1:]], len(tests[labels[0]]),
                         np.concatenate([tests[l].p_one_sided.to_numpy() for l in labels[1:]]))
        LADDER_ROWS.append({'dataset': DS_NAME[ds], 'row': row, 'calls': len(calls[labels[0]]),
                            'calls (redundancy clusters)': CLUSTERS[ds].reindex(sorted(calls[labels[0]])).nunique(),
                            'relabeled calls': ', '.join(str(len(calls[l])) for l in labels[1:]),
                            'NCDR': rule['NCDR'], 'tested': len(tests[labels[0]]), 'calibration share': rule['calibration share'],
                            'pass': rule['pass'], 'df1, df2': f"{int(lr['df1'])}, {df2c}/{df2r}"})
LADDER = pd.DataFrame(LADDER_ROWS)
for ds in DS_NAME:
    LADDER[LADDER.dataset.eq(DS_NAME[ds])].to_csv(tables / f'ladder_rows_{DS_NAME[ds]}.csv', index=False)
display(LADDER.round(3))

## 6 · R4 · Descriptors per specimen, replicated between specimens (notebook 58, NB)

- **Curves:** NB specimen curves (natural-log rate per library unit) on each coordinate's 101-point grid; on P they
  are notebook 71's cached curves (masked where the fit did not converge or is separated).
- **Scale:** 58's thresholds read on the natural-log scale (the NB note): amplitude 0.5, range 0.1, prominence 0.2,
  inner 0.2, |ρ| 0.9. Pre-specified sensitivity: log1p(10⁴ × rate), the old display scale.
- **Segment units:** logistic S1→S2 and S2→S3 transitions per specimen on the coordinate. A specimen whose transitions
  are out of order has no segment units, so no onsets, extrema or boundary shifts (58's rule); its slopes are kept.
- **r:** from specimen × label pseudobulk natural-log rates ln(max(Σy, 0.5) / Σlibrary); r_c from C.
- **(iv) onset order** in mouse S2 given r (partial Spearman, 2,000 gene bootstraps, seed 58): claim if the P value
  is ≥ 0.3 with its interval above 0 and the count-split value ≥ 0.2; abandon if both are < 0.1.
- **(i) within-segment slopes:** signed √LR of a linear position term per specimen × segment (≥ 30 units, fixed α);
  agreement = Spearman between the two specimens of a species over fold-protected genes; (i-b) inside C.
- **(ii) interior extrema** replicated in both specimens of a species.
- **Step nulls** on P and D: count rows with their library permuted inside specimen × C (arm A, seeds 5901+) or
  specimen × L (arm B, 6001+), 10 replicates each; every descriptor recomputed. An observed value counts as beyond
  steps only when it exceeds every replicate of both arms.

In [ ]:
def scaled_curve(curve, scale):
    c = np.asarray(curve, float)
    c = np.log1p(1e4 * np.exp(c)) if scale == 'log1p' else c
    return np.where(np.isfinite(c), c, 0.)          # a failed fit is a flat curve: never monotone, no extremum


def scaled_means(means, scale):
    m = np.asarray(means, float)
    return np.log1p(1e4 * np.exp(m)) if scale == 'log1p' else m


def descriptor_payload(v, counts, library, *, curves=None, coord_slopes=True):
    """Per-specimen descriptors on one coordinate (both scales): onsets, extrema, r, r_c, slopes in L and in C."""
    pos, spec, seg, cseg, grid = v['position'], v['specimen'], v['segment'], v['coord_segment'], v['grid']
    trans = logistic_transitions(pos, seg, spec).pivot(index='specimen', columns='transition', values='position')
    if curves is None:
        curves = nb_specimen_curves(counts, library, pos, spec, grid, v['knots'], v['alpha'], n_jobs=N_JOBS)
    slopes = nb_segment_slopes(counts, library, pos, spec, seg, v['alpha'], n_jobs=N_JOBS)
    slopes_c = nb_segment_slopes(counts, library, pos, spec, cseg, v['alpha'], n_jobs=N_JOBS) if coord_slopes else None
    means = pseudobulk_log_rates(counts, library, spec, seg)
    means_c = pseudobulk_log_rates(counts, library, spec, cseg)
    out = {'grid': np.asarray(grid, float), 'specimens': np.asarray(trans.index, str),
           't12': trans['S1→S2'].to_numpy(float), 't23': trans['S2→S3'].to_numpy(float)}
    skipped = []
    for name in out['specimens']:
        t12, t23 = trans.loc[name, 'S1→S2'], trans.loc[name, 'S2→S3']
        out[f'{name}|slope'] = slopes[name].to_numpy(np.float32)
        if slopes_c is not None:
            out[f'{name}|slope_c'] = slopes_c[name].to_numpy(np.float32)
        if not grid[0] < t12 < t23 < grid[-1]:
            skipped.append(name)
            continue
        units = segment_units(grid, t12, t23, grid[0], grid[-1])
        for scale in ('ln', 'log1p'):
            c = scaled_curve(curves[name], scale)
            onsets = monotone_onsets(c, units, min_amplitude=MIN_AMPLITUDE, min_abs_rho=MIN_ABS_RHO)
            for column in ('onset', 'direction', 'monotone', 'width'):
                out[f'{name}|{scale}|{column}'] = onsets[column].to_numpy(float)
            out[f'{name}|{scale}|extremum_segment'], out[f'{name}|{scale}|extremum_sign'] = interior_extrema(
                c, units, min_amplitude=MIN_AMPLITUDE, min_prominence=MIN_PROMINENCE, inner=INNER)
            out[f'{name}|{scale}|r'] = switch_fraction(scaled_means(means[name], scale), MIN_RANGE)
            out[f'{name}|{scale}|r_c'] = switch_fraction(scaled_means(means_c[name], scale), MIN_RANGE)
    out['skipped'] = np.asarray(skipped, str)
    return out


def onset_table(Dp, pair, scale):
    a, b = pair
    get = lambda n, c: Dp.get(f'{n}|{scale}|{c}', np.full(len(GENES), np.nan))   # noqa: E731
    return pd.DataFrame({'onset_a': get(a, 'onset'), 'onset_b': get(b, 'onset'), 'dir_a': get(a, 'direction'),
                         'dir_b': get(b, 'direction'), 'mono_a': get(a, 'monotone').astype(bool) if f'{a}|{scale}|monotone' in Dp else False,
                         'mono_b': get(b, 'monotone').astype(bool) if f'{b}|{scale}|monotone' in Dp else False,
                         'width_a': get(a, 'width'), 'width_b': get(b, 'width'), 'r_a': get(a, 'r'), 'r_b': get(b, 'r'),
                         'rc_a': get(a, 'r_c'), 'rc_b': get(b, 'r_c')}, index=GENES)


def ordering_test(table, segment_index, keep=None, covariates=('r_a', 'r_b'), boot=False):
    lo, hi = segment_index, segment_index + 1
    m = (table.mono_a & table.mono_b & table.dir_a.eq(table.dir_b) & table.onset_a.ge(lo) & table.onset_a.lt(hi)
         & table.onset_b.ge(lo) & table.onset_b.lt(hi) & table[list(covariates)].notna().all(axis=1))
    if keep is not None:
        m &= keep
    t = table[m]
    row = {'genes': int(m.sum()), 'partial Spearman (onset | r)': np.nan, '95% low': np.nan, '95% high': np.nan}
    if len(t) >= 20:
        if boot:
            est, low, high = bootstrap_partial_spearman(t.onset_a, t.onset_b, t[list(covariates)].to_numpy(),
                                                        n_boot=N_BOOT, seed=BOOT_SEED)
            row.update({'partial Spearman (onset | r)': est, '95% low': low, '95% high': high,
                        'Spearman onset a vs b': spearmanr(t.onset_a, t.onset_b).statistic})
        else:
            row['partial Spearman (onset | r)'] = partial_spearman(t.onset_a, t.onset_b, t[list(covariates)].to_numpy())
    return row, t


def slope_agreement(Dp, pair, keep, key='slope', min_genes=MIN_SLOPE_GENES):
    a, b = Dp.get(f'{pair[0]}|{key}'), Dp.get(f'{pair[1]}|{key}')
    out = {}
    for k, seg in enumerate(('S1', 'S2', 'S3')):
        if a is None or b is None:
            out[seg] = np.nan
            continue
        m = keep & np.isfinite(a[:, k]) & np.isfinite(b[:, k])
        out[seg] = spearmanr(a[m, k], b[m, k]).statistic if m.sum() > min_genes else np.nan
    return out


def replicated_extrema(Dp, pair, scale):
    a, b = pair
    if f'{a}|{scale}|extremum_segment' not in Dp or f'{b}|{scale}|extremum_segment' not in Dp:
        return np.zeros(len(GENES), bool), {s: 0 for s in ('S1', 'S2', 'S3')}
    sa, sb = Dp[f'{a}|{scale}|extremum_segment'], Dp[f'{b}|{scale}|extremum_segment']
    ga, gb = Dp[f'{a}|{scale}|extremum_sign'], Dp[f'{b}|{scale}|extremum_sign']
    same = (sa >= 0) & (sa == sb) & (ga == gb)
    return same, {seg: int((same & (sa == k)).sum()) for k, seg in enumerate(('S1', 'S2', 'S3'))}


def descriptor_metrics(Dp, keep, scale='ln', boot=False):
    row = {}
    t = onset_table(Dp, MICE, scale)
    r_iv, _ = ordering_test(t, 1, keep=keep, boot=boot)
    row['(iv) mouse S2 partial ρ'] = r_iv['partial Spearman (onset | r)']
    row['(iv) mouse S2 genes'] = r_iv['genes']
    if boot:
        row['(iv) 95% low'], row['(iv) 95% high'] = r_iv['95% low'], r_iv['95% high']
    r_ivc, _ = ordering_test(t, 1, keep=keep, covariates=('rc_a', 'rc_b'))
    row['(iv-b) mouse S2 partial ρ given coordinate-segment r'] = r_ivc['partial Spearman (onset | r)']
    for sp, pair in PAIRS.items():
        for seg, value in slope_agreement(Dp, pair, keep).items():
            row[f'(i) {sp} {seg} slope ρ'] = value
        for seg, value in slope_agreement(Dp, pair, keep, 'slope_c').items():
            row[f'(i-b) {sp} {seg} slope ρ within coordinate segments'] = value
        _, counts = replicated_extrema(Dp, pair, scale)
        for seg, value in counts.items():
            row[f'(ii) {sp} {seg} replicated extrema'] = value
    return row


DESC = {}
p_curves = {n: np.where((M71[f'converged|{n}'].astype(bool) & ~M71[f'separated|{n}'].astype(bool))[:, None],
                        M71[f'curve|{n}'], np.nan) for n in (*MICE, *HUMANS)}
for code, v in V.items():
    DESC[code] = cached(f'descriptors_{code}', lambda v=v, code=code: descriptor_payload(
        v, v['counts'], v['library'], curves=p_curves if code == 'P' else None),
        params={'rules': [MIN_AMPLITUDE, MIN_ABS_RHO, MIN_RANGE, MIN_PROMINENCE, INNER], 'basis_df': BASIS_DF},
        inputs={**v['key'], 'alpha': v['alpha']['alpha'], 'coord_segment': v['coord_segment'],
                'curves71': digest([p_curves[n] for n in sorted(p_curves)]) if code == 'P' else None})
    if len(DESC[code]['skipped']):
        READING.append(f"R4: on {code} the logistic transitions are out of order in {list(DESC[code]['skipped'])} (72's C1 "
                       'is not met for the same reason on P), so those specimens have no segment units: no onsets, extrema or '
                       'boundary shifts there (58\'s rule for a specimen without ordered transitions). Their slopes need no '
                       'transitions and are kept.')
SKIPPED_P = [str(n) for n in DESC['P']['skipped']]


def null_replicate(code, arm, rep):
    v = V[code]
    blocks = np.array([f'{a}|{b}' for a, b in zip(v['specimen'], v['coord_segment'] if arm == 'A' else v['segment'])])
    index = blockwise_permutation(blocks, np.random.default_rng(NULL_SEEDS_58[arm] + rep))
    Dn = descriptor_payload(v, v['counts'][index], v['library'][index])
    out = {f'{scale}|{k}': np.array(val, float) for scale in ('ln', 'log1p')
           for k, val in descriptor_metrics(Dn, FOLD_PROTECTED[code], scale).items()}
    out.update({k: val for k, val in Dn.items() if k.endswith('|slope') or k.endswith('|slope_c')})
    return out


NULLS = {}
for code in ('P', 'D'):
    v = V[code]
    for arm in 'AB':
        for rep in range(1, N_REPLICATES + 1):
            NULLS[(code, arm, rep)] = cached(
                f'null58_{code}_{arm}_{rep}', lambda code=code, arm=arm, rep=rep: null_replicate(code, arm, rep),
                params={'arm': arm, 'rep': rep, 'seed': NULL_SEEDS_58[arm],
                        'rules': [MIN_AMPLITUDE, MIN_ABS_RHO, MIN_RANGE, MIN_PROMINENCE, INNER]},
                inputs={**v['key'], 'alpha': v['alpha']['alpha'], 'coord_segment': v['coord_segment'],
                        'protected': digest(FOLD_PROTECTED[code])})

OBSERVED, COMPARISON = {}, {}
for code in V:
    for scale in ('ln', 'log1p'):
        OBSERVED[(code, scale)] = pd.Series(descriptor_metrics(DESC[code], FOLD_PROTECTED[code], scale,
                                                               boot=(scale == 'ln' and code != 'H')))
for code in ('P', 'D'):
    for scale in ('ln', 'log1p'):
        frame = pd.DataFrame([{**{k.split('|', 1)[1]: float(val) for k, val in NULLS[(code, arm, rep)].items()
                                  if k.startswith(scale + '|')}, 'arm': arm, 'rep': rep}
                              for arm in 'AB' for rep in range(1, N_REPLICATES + 1)])
        obs = OBSERVED[(code, scale)]
        cmp = pd.DataFrame({'observed': obs.reindex(frame.columns.drop(['arm', 'rep'])),
                            'arm A max': frame[frame.arm.eq('A')].drop(columns=['arm', 'rep']).max(),
                            'arm A median': frame[frame.arm.eq('A')].drop(columns=['arm', 'rep']).median(),
                            'arm B max': frame[frame.arm.eq('B')].drop(columns=['arm', 'rep']).max(),
                            'arm B median': frame[frame.arm.eq('B')].drop(columns=['arm', 'rep']).median()})
        cmp['exceeds every null replicate'] = cmp.observed > cmp[['arm A max', 'arm B max']].max(axis=1)
        COMPARISON[(code, scale)] = cmp
        frame.assign(coordinate=code, scale=scale).to_csv(tables / f'step_nulls_{code}_{scale}.csv', index=False)
STEP_NULLS = pd.concat({f'{c} · {s}': t for (c, s), t in COMPARISON.items()}, names=['coordinate · scale', 'metric'])
STEP_NULLS.to_csv(tables / 'step_nulls.csv')
display(COMPARISON[('P', 'ln')].round(3))

# (iv) onset order: per coordinate, the count-split arm and the decision
ORDER_ROWS, IV_GENES = [], {}
for code in V:
    if code == 'H':
        continue
    t = onset_table(DESC[code], MICE, 'ln')
    row, genes_iv = ordering_test(t, 1, keep=FOLD_PROTECTED[code], boot=True)
    IV_GENES[code] = genes_iv
    genes_iv.to_csv(tables / f'onsets_iv_primary_{code}.csv')
    ORDER_ROWS.append({'arm': code, 'pair': 'mouse', 'segment': 'S2', 'genes used': 'fold-protected', **row})
    for sp, seg_index in (('human', 1), ('human', 2)):
        r_h, _ = ordering_test(onset_table(DESC[code], HUMANS, 'ln'), seg_index, keep=FOLD_PROTECTED[code], boot=True)
        ORDER_ROWS.append({'arm': code, 'pair': 'human (one donor)', 'segment': f'S{seg_index + 1}',
                           'genes used': 'fold-protected', **r_h})
split_value = np.nan
if 'H' in V:
    t = onset_table(DESC['H'], MICE, 'ln')
    split_row, _ = ordering_test(t, 1, boot=True)
    split_value = split_row['partial Spearman (onset | r)']
    ORDER_ROWS.append({'arm': 'count split (coordinate from half A, data from half B)', 'pair': 'mouse', 'segment': 'S2',
                       'genes used': 'all', **split_row})
    row, _ = ordering_test(t, 1, keep=~pd.Index(GENES).isin(sorted(SPLIT_CONSTRUCTION)), boot=True)
    ORDER_ROWS.append({'arm': 'count split (coordinate from half A, data from half B)', 'pair': 'mouse', 'segment': 'S2',
                       'genes used': 'outside split-A integration features', **row})
ORDERING = pd.DataFrame(ORDER_ROWS)
ORDERING.to_csv(tables / 'onsets_iv.csv', index=False)
primary_iv = ORDERING[ORDERING.arm.eq('P') & ORDERING.pair.eq('mouse')].iloc[0]
iv_value, iv_low = primary_iv['partial Spearman (onset | r)'], primary_iv['95% low']
if iv_value >= CLAIM_PRIMARY and iv_low > 0 and split_value >= CLAIM_SPLIT:
    DECISION_IV = 'claim: onset order inside mouse S2 replicates between mice beyond r'
elif iv_value < ABANDON and split_value < ABANDON:
    DECISION_IV = 'abandon: onset order adds nothing reproducible beyond r'
else:
    DECISION_IV = 'inconclusive'
IV_BEYOND_NULLS = bool(COMPARISON[('P', 'ln')].loc['(iv) mouse S2 partial ρ', 'exceeds every null replicate'])
display(ORDERING.round(3))
print('(iv) decision:', DECISION_IV, '· beyond every step-null replicate:', IV_BEYOND_NULLS)

# (i) within-segment gradients: observed per coordinate, count split, step nulls
GRADIENT_ROWS = []
split_rho = {sp: slope_agreement(DESC['H'], pair, np.ones(len(GENES), bool)) for sp, pair in PAIRS.items()} if 'H' in V else {}
for sp, pair in PAIRS.items():
    for seg in ('S1', 'S2', 'S3'):
        key = f'(i) {sp} {seg} slope ρ'
        row = {'species': sp, 'segment': seg}
        for code in V:
            if code != 'H':
                row[f'{code} slope ρ'] = OBSERVED[(code, 'ln')][key]
        row['count-split slope ρ'] = split_rho.get(sp, {}).get(seg, np.nan)
        for code in ('P', 'D'):
            cmp = COMPARISON[(code, 'ln')]
            row[f'{code} step-null max'] = cmp.loc[key, ['arm A max', 'arm B max']].max()
            row[f'{code} beyond every step null'] = bool(cmp.loc[key, 'exceeds every null replicate'])
            row[f'{code} (i-b) slope ρ in C'] = OBSERVED[(code, 'ln')][f'(i-b) {sp} {seg} slope ρ within coordinate segments']
            row[f'{code} (i-b) beyond every step null'] = bool(
                cmp.loc[f'(i-b) {sp} {seg} slope ρ within coordinate segments', 'exceeds every null replicate'])
        GRADIENT_ROWS.append(row)
GRADIENTS = pd.DataFrame(GRADIENT_ROWS)
slope_cols = {}
for code in V:
    for n in (*MICE, *HUMANS):
        for k, seg in enumerate(('S1', 'S2', 'S3')):
            for key, kind in (('slope', 'label'), ('slope_c', 'coordinate')):
                if f'{n}|{key}' in DESC[code]:
                    slope_cols[f'{code} {n} {seg} {kind}'] = DESC[code][f'{n}|{key}'][:, k]
pd.DataFrame(slope_cols, index=GENES).assign(fold_protected=FOLD_PROTECTED['P']).to_csv(tables / 'slopes_i_per_gene.csv')
display(GRADIENTS.round(3))

# (ii) interior extrema: replicated per coordinate; cross-coordinate rule (>= 2 of P, D, E including E); step nulls
EXTREMA_ROWS = []
for code in ('P', 'D', 'E'):
    if code not in DESC:
        continue
    for sp, pair in PAIRS.items():
        same, _ = replicated_extrema(DESC[code], pair, 'ln')
        seg_index = DESC[code].get(f'{pair[0]}|ln|extremum_segment')
        sign = DESC[code].get(f'{pair[0]}|ln|extremum_sign')
        for i in np.flatnonzero(same):
            EXTREMA_ROWS.append({'coordinate': code, 'gene': GENES[i], 'species': sp,
                                 'segment': ('S1', 'S2', 'S3')[int(seg_index[i])], 'kind': 'peak' if sign[i] > 0 else 'trough',
                                 'fold-protected': bool(FOLD_PROTECTED[code][i])})
EXTREMA = pd.DataFrame(EXTREMA_ROWS, columns=['coordinate', 'gene', 'species', 'segment', 'kind', 'fold-protected'])
EXTREMA.to_csv(tables / 'extrema_ii.csv', index=False)
keyed = {code: set(zip(f.gene, f.species, f.segment, f.kind)) for code, f in EXTREMA.groupby('coordinate')}
cross = pd.DataFrame([{'gene': g, 'species': s, 'segment': seg, 'kind': kind, **{c: (g, s, seg, kind) in keyed.get(c, set())
                                                                              for c in ('P', 'D', 'E')}}
                      for g, s, seg, kind in set().union(*keyed.values())]) if keyed else pd.DataFrame()
if len(cross):
    cross['coordinates'] = cross[['P', 'D', 'E']].sum(axis=1)
    cross['counts (>= 2 of P, D, E including E)'] = cross.coordinates.ge(2) & cross.E
    cross.sort_values(['species', 'segment', 'gene']).to_csv(tables / 'extrema_ii_across_coordinates.csv', index=False)
EXTREMA_CELLS = pd.DataFrame([{'species': sp, 'segment': seg,
                               **{f'{code} replicated': OBSERVED[(code, 'ln')][f'(ii) {sp} {seg} replicated extrema']
                                  for code in ('P', 'D', 'E') if code in DESC},
                               **{f'{code} step-null max': COMPARISON[(code, 'ln')].loc[f'(ii) {sp} {seg} replicated extrema',
                                                                                       ['arm A max', 'arm B max']].max()
                                  for code in ('P', 'D')},
                               **{f'{code} beyond every step null': bool(COMPARISON[(code, 'ln')].loc[
                                   f'(ii) {sp} {seg} replicated extrema', 'exceeds every null replicate']) for code in ('P', 'D')},
                               'cross-coordinate rule genes': int(cross[(cross.species == sp) & (cross.segment == seg)][
                                   'counts (>= 2 of P, D, E including E)'].sum()) if len(cross) else 0}
                              for sp in PAIRS for seg in ('S1', 'S2', 'S3')])
display(EXTREMA_CELLS)

SCALE_SENSITIVITY = pd.DataFrame({f'{code} · {scale}': OBSERVED[(code, scale)] for code in ('P', 'D') for scale in ('ln', 'log1p')})
for code in ('P', 'D'):
    SCALE_SENSITIVITY[f'{code} · log1p beyond every step null'] = COMPARISON[(code, 'log1p')]['exceeds every null replicate']
    SCALE_SENSITIVITY[f'{code} · ln beyond every step null'] = COMPARISON[(code, 'ln')]['exceeds every null replicate']
SCALE_SENSITIVITY.to_csv(tables / 'scale_sensitivity.csv')
pd.DataFrame({code: OBSERVED[(code, 'ln')] for code in V}).to_csv(tables / 'descriptors_summary.csv')

## 7 · R7 · Solute-transporter micro-gradients: the gene tests (new; Figure 3b)

**Gene set T.** HGNC gene-group branches "Solute carriers" (752) and "ATP binding cassette transporters" (417),
protein-coding, downloaded 2026-10-12 (sha256 asserted), mapped by the export's `human_symbol` into notebook 71's
universe. Branch membership, not a symbol pattern. SLC25 (mitochondrial carriers) is its own family, and every T
result is reported with and without it (O5).

**Within-species NB tests on P** (each species' own units; genes with counts in both of its specimens; moderation and
BH over those genes):

| Test | Numerator | Denominator | df₁ / df₂ |
|---|---|---|---|
| W1 positional | 1 + specimen contrast → + spline | specimen contrast × spline | 6 / 6 |
| W2 beyond L-steps | 1 + specimen contrast + L-steps → + spline | + specimen × steps → + specimen × spline | 6 / 6 |
| W3 beyond C-steps | as W2 with C | as W2 with C | 6 / 6 |

A **micro-gradient gene** has q ≤ 0.05 in both W2 and W3 (converged, not separated). **Localisation:** a segment in
which both specimens' NB slope statistics share a sign with |·| ≥ 2.

**Exact nulls.** W3 under arm A and W2 under arm B (R4's permutations, seeds 5901+ and 6001+): calibrated when the
pooled share of p ≤ 0.05 is ≤ 0.075.

In [ ]:
HGNC_ROWS = []
for group_id, (group_name, checksum) in HGNC.items():
    path = data_root / 'external' / 'hgnc' / f'hgnc_group_{group_id}.tsv'
    assert sha256(path.read_bytes()).hexdigest() == checksum, f'{path.name} changed since {HGNC_DOWNLOADED}'
    table = pd.read_csv(path, sep='\t')
    table = table[table['Locus type'].eq('gene with protein product') & table.Status.eq('Approved')]
    HGNC_ROWS.append(table.assign(branch=group_name, branch_id=group_id))
HGNC_TABLE = pd.concat(HGNC_ROWS, ignore_index=True)
human_of = pd.Series(U.var.human_symbol.astype(str).to_numpy(), index=EXPORT_GENES)
hgnc_of = HGNC_TABLE.drop_duplicates('Approved symbol').set_index('Approved symbol')
T_TABLE = pd.DataFrame({'gene': GENES, 'human_symbol': human_of.reindex(GENES).to_numpy()})
T_TABLE = T_TABLE[T_TABLE.human_symbol.isin(hgnc_of.index)].copy()
T_TABLE['branch'] = hgnc_of.branch.reindex(T_TABLE.human_symbol).to_numpy()
T_TABLE['HGNC group'] = hgnc_of['Group name'].reindex(T_TABLE.human_symbol).to_numpy()
T_TABLE['family'] = np.where(T_TABLE['HGNC group'].astype(str).str.contains('family 25$', regex=True),
                             'SLC25 (mitochondrial carriers)',
                             np.where(T_TABLE.branch.eq('Solute carriers'), 'other solute carriers', 'ABC transporters'))
T_TABLE = T_TABLE.set_index('gene')
T_GENES = list(T_TABLE.index)
T_NO25 = list(T_TABLE.index[~T_TABLE.family.str.startswith('SLC25')])
print(f'T: {len(T_GENES)} genes in 71\'s universe ({(T_TABLE.family == "SLC25 (mitochondrial carriers)").sum()} SLC25, '
      f'{(T_TABLE.family == "ABC transporters").sum()} ABC); without SLC25: {len(T_NO25)}')

W_TESTS = {'W1': None, 'W2': 'L', 'W3': 'C'}
W_OK = {sp: np.ones(len(GENES), bool) for sp in PAIRS}
for sp, pair in PAIRS.items():
    for n in pair:
        rows = P['specimen'] == n
        W_OK[sp] &= np.asarray(P['counts'][rows].sum(axis=0)).ravel() > 0


def w_lr(sp, test, counts, library):
    """One within-species test on P's units of species ``sp`` (genes with counts in both specimens)."""
    rows = np.flatnonzero(P['human'] == (sp == 'human'))
    ok = W_OK[sp]
    segments = {'W1': None, 'W2': P['segment'][rows], 'W3': P['coord_segment'][rows]}[test]
    designs, comparisons, weights = within_species_designs(P['position'][rows], P['specimen'][rows], P['knots'], segments)
    alpha = {'alpha': ALPHA71['alpha'][ok], 'converged': ALPHA71['converged'][ok]}
    return lr_payload(nested_lr(counts[rows][:, ok], library[rows], designs, comparisons, weights, alpha, n_jobs=N_JOBS))


def w_frame(sp, lr, counts):
    rows = np.flatnonzero(P['human'] == (sp == 'human'))
    ok = W_OK[sp]
    covariate = np.log(np.asarray(counts[rows][:, ok].mean(axis=0)).ravel())
    frame = moderated_f(pd.Series(lr['numerator'], index=pd.Index(GENES)[ok]), int(lr['df1']),
                        np.asarray(lr['denominator'], float), int(lr['df2']), covariate=covariate, robust=False)
    frame['q'] = bh(frame.p.to_numpy())
    frame['converged'] = np.asarray(lr['converged'], bool)
    frame['separated'] = np.asarray(lr['separated'], bool)
    out = frame.reindex(GENES)
    out['converged'] = pd.Series(False, index=GENES).mask(ok, frame['converged'].reindex(GENES)).astype(bool)
    out['separated'] = pd.Series(True, index=GENES).mask(ok, frame['separated'].reindex(GENES)).astype(bool)
    return out


W = {}
for sp in PAIRS:
    for test in W_TESTS:
        lr = cached(f'w_{test}_{sp}', lambda sp=sp, test=test: w_lr(sp, test, P['counts'], P['library']),
                    params={'test': test, 'species': sp, 'basis_df': BASIS_DF},
                    inputs={**P['key'], 'alpha': ALPHA71['alpha'], 'coord_segment': P['coord_segment']})
        assert (int(lr['df1']), int(lr['df2'])) == (6, 6), (sp, test, int(lr['df1']), int(lr['df2']))
        W[(sp, test)] = w_frame(sp, lr, P['counts'])


def w_null(arm, rep):
    blocks = np.array([f'{a}|{b}' for a, b in zip(P['specimen'], P['coord_segment'] if arm == 'A' else P['segment'])])
    index = blockwise_permutation(blocks, np.random.default_rng(NULL_SEEDS_58[arm] + rep))
    counts, library = P['counts'][index], P['library'][index]
    test = 'W3' if arm == 'A' else 'W2'                      # the exact null of each test
    out = {}
    for sp in PAIRS:
        frame = w_frame(sp, w_lr(sp, test, counts, library), counts)
        out[f'{sp}|p'] = frame.p.to_numpy(float)
        out[f'{sp}|ok'] = (frame.converged & ~frame.separated).to_numpy(bool)
    return out


W_NULLS = {(arm, rep): cached(f'w_null_{arm}_{rep}', lambda arm=arm, rep=rep: w_null(arm, rep),
                              params={'arm': arm, 'rep': rep, 'seed': NULL_SEEDS_58[arm]},
                              inputs={**P['key'], 'alpha': ALPHA71['alpha'], 'coord_segment': P['coord_segment']})
           for arm in 'AB' for rep in range(1, N_REPLICATES + 1)}
CAL_ROWS = []
for arm, test in (('A', 'W3'), ('B', 'W2')):
    for sp in PAIRS:
        p = np.concatenate([W_NULLS[(arm, rep)][f'{sp}|p'][W_NULLS[(arm, rep)][f'{sp}|ok']] for rep in range(1, N_REPLICATES + 1)])
        CAL_ROWS.append({'test': test, 'exact null': f'arm {arm}', 'species': sp, 'p-values': len(p),
                         'share p ≤ 0.05': float(np.mean(p <= .05))})
    pooled = np.concatenate([W_NULLS[(arm, rep)][f'{sp}|p'][W_NULLS[(arm, rep)][f'{sp}|ok']]
                             for rep in range(1, N_REPLICATES + 1) for sp in PAIRS])
    CAL_ROWS.append({'test': test, 'exact null': f'arm {arm}', 'species': 'both (pooled)', 'p-values': len(pooled),
                     'share p ≤ 0.05': float(np.mean(pooled <= .05))})
W_CALIBRATION = pd.DataFrame(CAL_ROWS)
W_CALIBRATION['calibrated (≤ 0.075)'] = W_CALIBRATION['share p ≤ 0.05'] <= CALIBRATION_MAX
W_CALIBRATION.to_csv(tables / 'transporter_calibration.csv', index=False)
W_CALIBRATED = bool(W_CALIBRATION[W_CALIBRATION.species.eq('both (pooled)')]['calibrated (≤ 0.075)'].all())
display(W_CALIBRATION.round(4))

for sp in PAIRS:
    good = W[(sp, 'W2')].converged & ~W[(sp, 'W2')].separated & W[(sp, 'W3')].converged & ~W[(sp, 'W3')].separated
    for test in W_TESTS:
        T_TABLE[f'{test} z {sp}'] = W[(sp, test)].z.reindex(T_TABLE.index)
        T_TABLE[f'{test} q {sp}'] = W[(sp, test)].q.reindex(T_TABLE.index)
    micro = good & W[(sp, 'W2')].q.le(ALPHA) & W[(sp, 'W3')].q.le(ALPHA)
    T_TABLE[f'micro-gradient {sp}'] = micro.reindex(T_TABLE.index).fillna(False).astype(bool)
    T_TABLE[f'zonated (W1 q ≤ 0.05) {sp}'] = W[(sp, 'W1')].q.reindex(T_TABLE.index).le(ALPHA)
    for k, seg in enumerate(('S1', 'S2', 'S3')):
        a, b = (DESC['P'][f'{n}|slope'][:, k] for n in PAIRS[sp])
        local = (np.sign(a) == np.sign(b)) & (np.abs(a) >= LOCAL_SLOPE) & (np.abs(b) >= LOCAL_SLOPE)
        T_TABLE[f'local gradient {sp} {seg}'] = pd.Series(local, index=GENES).reindex(T_TABLE.index).astype(bool)
ALL_MICRO = {sp: int((W[(sp, 'W2')].q.le(ALPHA) & W[(sp, 'W3')].q.le(ALPHA)
                      & W[(sp, 'W2')].converged & ~W[(sp, 'W2')].separated
                      & W[(sp, 'W3')].converged & ~W[(sp, 'W3')].separated).sum()) for sp in PAIRS}
T_TABLE.to_csv(tables / 'transporters.csv')
W_GENES = pd.concat({f'{test} {sp}': W[(sp, test)][['F', 'z', 'q', 'converged', 'separated']] for (sp, test) in W}, axis=1)
W_GENES.to_csv(tables / 'within_species_tests_genes.csv')
print('Micro-gradient genes (W2 and W3 q ≤ 0.05), all universe genes:', ALL_MICRO)

## 8 · R6 (part 1) · The placebo coordinate and WSR

**Placebo** (72's section 8; phase-3 §2.7's control, "any within-segment metric must beat the placebo coordinate,
otherwise it is measuring co-expression programs, not position"):
1. Eligible genes: 71's universe outside the base integration features and the 15 panel and anchor genes, with W1
   converged and not separated in both species. They are ranked by the larger of the two species' W1 z (R7).
2. Within each decile of log mean count (deciles of the base integration features), the lowest-ranked eligible genes
   are taken, as many as there are base integration features in that decile (expression-matched, as notebook 53).
3. The package, unmodified, reruns stages 03–04 once with every other gene flagged `exclude_from_harmony_hvg`
   (`package_refits`, 72's route, default threads as notebooks 70 and 72 ran). Guard: the refit's integration features
   are placebo genes. The placebo's S1 → S3 order is reported and never required; a global flip changes neither slope
   agreement nor WSR.
4. **One parameter differs (read here; the protocol did not foresee it).** The package keeps only variable genes as
   integration features (Seurat dispersion ≥ 0.5 in each species). Non-zonal genes are by construction not variable:
   with the shipped value only 45 placebo genes qualified and stage 03 stopped (50 PCs need ≥ 50 features). Notebook
   53's placebo used its pipeline with the placebo genes as the features, so the closest reading keeps the package
   and its tracked configuration and sets `integration.hvg_min_disp` to −10⁹ for this refit only, through the
   configuration's documented `parameters` section: every placebo gene inside the package's mean range (0.0125–3)
   becomes a feature.

**WSR** (notebook 53's statistic, unchanged and log-normalized: held-out residual transfer is in P4c's class, 72's
O3): for each species, fold and direction, held-out genes are residualized in each specimen on its own segment +
coverage quadratic oracle; a per-gene, per-segment spline (df 4) of the training specimen's fold-k coordinate
predicts the test specimen's residuals; the statistic is the mean atanh(r) over genes; the null is 500 within-segment
permutations of the test coordinate (seed 61). Held-out genes are 72's evaluation genes (its rule: measured, not an
anchor, not a label/orientation gene, detection ≥ 10% in the species, external level ≥ 1); the placebo's held-out
genes exclude the placebo set.

In [ ]:
if not visium_root:
    raise RuntimeError('The package configs expand $PSEUDOSPACE_VISIUM_ROOT: set it or pass --visium-root (notebook 70).')
os.environ['PSEUDOSPACE_VISIUM_ROOT'] = str(Path(visium_root).expanduser().resolve())
w1_max = pd.concat([W[(sp, 'W1')].z for sp in PAIRS], axis=1).max(axis=1)
w1_good = np.logical_and.reduce([W[(sp, 'W1')].converged.to_numpy(bool) & ~W[(sp, 'W1')].separated.to_numpy(bool)
                                 for sp in PAIRS])
eligible = pd.Series(w1_good & ~pd.Index(GENES).isin(sorted(BASE_FEATURES | set(FIXED_GENES))), index=GENES)
LMC_ALL = pd.Series(np.log(np.maximum(np.asarray(X_ALL[P['units']].mean(axis=0)).ravel(), 1e-12)), index=EXPORT_GENES)
features = sorted(BASE_FEATURES & set(EXPORT_GENES))
edges = np.quantile(LMC_ALL[features], np.linspace(.1, .9, 9))
feature_decile = np.searchsorted(edges, LMC_ALL[features].to_numpy(), side='right')
candidate = pd.DataFrame({'lmc': LMC_ALL.reindex(GENES).to_numpy(), 'w1 max z': w1_max.reindex(GENES).to_numpy()},
                         index=GENES)[eligible.to_numpy()]
candidate['decile'] = np.searchsorted(edges, candidate.lmc.to_numpy(), side='right')
PLACEBO_ROWS, PLACEBO = [], []
for k in range(10):
    need = int((feature_decile == k).sum())
    pool = candidate[candidate.decile.eq(k)].sort_values(['w1 max z', 'lmc'])
    take = list(pool.index[:need])
    PLACEBO += take
    PLACEBO_ROWS.append({'decile': k, 'base integration features': need, 'eligible genes': len(pool), 'taken': len(take),
                         'shortfall': need - len(take), 'max W1 z taken': pool['w1 max z'].iloc[:need].max() if len(take) else np.nan})
PLACEBO = sorted(PLACEBO)
PLACEBO_GENES_TABLE = pd.DataFrame(PLACEBO_ROWS)
PLACEBO_GENES_TABLE.to_csv(tables / 'placebo_genes_by_decile.csv', index=False)
candidate.assign(placebo=candidate.index.isin(PLACEBO)).to_csv(tables / 'placebo_genes.csv')
display(PLACEBO_GENES_TABLE.round(3))
print(f'Placebo set: {len(PLACEBO)} genes against {len(features)} base integration features')


PL_CONFIG = REFIT_DIR / 'placebo_config'
PL_CONFIG.mkdir(parents=True, exist_ok=True)
for name in ('samples.csv', 'marker_panels.yaml'):
    (PL_CONFIG / name).write_text((CONFIG_HVM / name).read_text())
pl_dataset = yaml.safe_load((CONFIG_HVM / 'dataset.yaml').read_text())
pl_dataset['parameters'].setdefault('integration', {})['hvg_min_disp'] = -1.0e9
(PL_CONFIG / 'dataset.yaml').write_text('# Notebook 74 placebo refit: the tracked human vs mouse config with integration.hvg_min_disp = -1e9\n'
                                       + yaml.safe_dump(pl_dataset, sort_keys=False))
READING.append('Placebo refit: integration.hvg_min_disp is set to -1e9 for this refit only (the tracked configuration '
               'otherwise), because the package keeps only variable genes as integration features and the non-zonal '
               'placebo genes are not variable (45 of them qualified with the shipped 0.5 and stage 03 stopped).')


def placebo_refit():
    import shutil

    free = refit_module.free_gb(output)
    if free < 40:
        raise RuntimeError(f'Only {free:.0f} GB free; a refit needs >= 40 GB (72\'s rule).')
    folder = REFIT_DIR / 'placebo'
    if folder.exists():
        shutil.rmtree(folder)
    t0 = time.perf_counter()
    base = ad.read_h5ad(RUN['hvm'] / '02_qc' / 'units_qc.h5ad')
    keep = set(PLACEBO)
    units = refit_module.perturbed_units(base, exclude_genes=[g for g in base.var_names.astype(str) if g not in keep])
    del base
    refit_module.write_refit_input(units, RUN['hvm'], folder, {'name': 'placebo', 'genes': digest(PLACEBO),
                                                               'notebook': NOTEBOOK_LOGIC_VERSION})
    del units
    refit_module.run_stages_03_04(PL_CONFIG, data_root, folder, threads=None)
    payload = refit_module.extract_refit(folder)
    payload['freed_bytes'] = np.array(refit_module.cleanup_refit(folder))
    payload['seconds'] = np.array(time.perf_counter() - t0)
    return payload


PL_REFIT = cached_payload('placebo_refit', placebo_refit, root=cache,
                          params={'genes': PLACEBO, 'threads': 'default', 'hvg_min_disp': -1.0e9},
                          inputs={'units_qc': RUN['hvm'] / '02_qc' / 'units_qc.h5ad',
                                  'manifest_03': RUN['hvm'] / '03_integration' / 'manifest.json',
                                  'config': [PL_CONFIG / f for f in ('samples.csv', 'dataset.yaml', 'marker_panels.yaml')],
                                  'package': EXTERNAL},
                          code=digest(['74.placebo_refit.1', Path(refit_module.__file__)]))
PL_FEATURES = set(PL_REFIT['pass1_features'].astype(str)) | set(PL_REFIT['pass2_features'].astype(str))
assert PL_FEATURES <= set(PLACEBO), f'non-placebo genes among the placebo integration features: {sorted(PL_FEATURES - set(PLACEBO))[:5]}'
z_pl = refit_module.refit_position(PL_REFIT, IDS)
pl_s1 = np.nanmedian(z_pl[SEG_ALL == 'PT-S1'])
pl_s3 = np.nanmedian(z_pl[SEG_ALL == 'PT-S3'])
PL_FLIPPED = bool(pl_s1 > pl_s3)
if PL_FLIPPED:
    z_pl = -z_pl
PL_RECORD = {'pass-2 units': len(PL_REFIT['unit']), 'PT units with a position': int(np.isfinite(z_pl).sum()),
             'integration features (pass 1 / 2)': f"{len(PL_REFIT['pass1_features'])} / {len(PL_REFIT['pass2_features'])}",
             'placebo genes used as features': len(PL_FEATURES), 'placebo genes': len(PLACEBO),
             'flipped': PL_FLIPPED, 'runs S1 → S3 in every specimen': refit_module.runs_s1_to_s3(z_pl, SPEC_ALL, SEG_ALL),
             'Spearman with P (all units with both)': float(spearmanr(z_pl, RAW_ALL, nan_policy='omit').statistic),
             'seconds': float(PL_REFIT['seconds'])}
medians_pl = pd.DataFrame({'z': z_pl, 'specimen': SPEC_ALL, 'segment': SEG_ALL}).dropna().groupby(
    ['specimen', 'segment']).z.median().unstack()
V['PL'] = view(z_pl, 'PL', alpha=ALPHA71)
FOLD_PROTECTED['PL'] = ~pd.Index(GENES).isin(sorted(PL_FEATURES | set(FIXED_GENES)))
DESC['PL'] = cached('descriptors_PL_slopes', lambda: {f'{n}|slope': s.to_numpy(np.float32) for n, s in nb_segment_slopes(
    V['PL']['counts'], V['PL']['library'], V['PL']['position'], V['PL']['specimen'], V['PL']['segment'], ALPHA71,
    n_jobs=N_JOBS).items()}, params={'kind': 'label slopes'}, inputs={**V['PL']['key'], 'alpha': ALPHA71['alpha']})
PL_SLOPES = {sp: slope_agreement(DESC['PL'], pair, FOLD_PROTECTED['PL']) for sp, pair in PAIRS.items()}
pd.Series(PL_RECORD).to_frame('placebo').to_csv(tables / 'placebo_refit.csv')
display(pd.Series(PL_RECORD).to_frame('placebo'))
display(medians_pl.round(3))
print('Placebo slope agreement (fold-protected):', {sp: {k: round(x, 3) for k, x in v.items()} for sp, v in PL_SLOPES.items()})

### WSR on P, D and the placebo

Notebook 72's human vs mouse fold-2 refit failed its orientation guard (NaN), so every arm uses folds 0 and 1, with
the same held-out genes. The units are those with a finite position in every fold refit used and in the placebo.

In [ ]:
from pseudospace.anchor_coordinate import census_segment_profiles, segment_contrast
from pseudospace.landmark_reconstruction import wsr_correlations, wsr_statistic
from pseudospace.within_segment_validation import fit_segment_covariate_oracle, predict_segment_covariate_oracle

GS72 = pd.read_csv(nb72 / 'tables' / 'gene_sets_hvm.csv', index_col=0).reindex(EXPORT_GENES)
external = data_root / 'external'


def census(label, segment_map, early):
    counts = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
    meta = pd.read_csv(external / 'census_pt_segments' / f'{label}_pt_segment_pseudobulk_meta.csv', index_col=0)
    return census_segment_profiles(counts, meta.loc[counts.index], segment_map, early=early).reindex(EXPORT_GENES)


mouse_sn = census('mouse', {'epithelial cell of proximal tubule segment 1': 'S1',
                            'epithelial cell of proximal tubule segment 2': 'S2',
                            'epithelial cell of proximal tubule segment 3': 'S3'}, ['S1', 'S2'])
human_sn = census('human', {'kidney proximal convoluted tubule epithelial cell': 'early',
                            'epithelial cell of proximal tubule segment 3': 'S3'}, None)
human_donors = sorted(set(human_sn.columns.get_level_values('donor')))
assert human_donors[1::2] == ['HCA_29_ref', 'HCA_55_ref', 'PRECISE-36'], human_donors
REFERENCE = {'mouse': segment_contrast(mouse_sn, sex='male'), 'human': segment_contrast(human_sn, donors=human_donors[1::2])}
for sp, column in (('mouse', 'contrast_mouse_sn_male'), ('human', 'contrast_human_sn_halfB')):
    both = REFERENCE[sp][0].notna() & GS72[column].notna()
    np.testing.assert_allclose(REFERENCE[sp][0][both], GS72.loc[both, column], rtol=1e-9, atol=1e-9)
measured = pd.Series(np.asarray(X_ALL.sum(axis=0)).ravel() > 0, index=EXPORT_GENES)
EVAL = {sp: (measured & ~GS72.anchor_loose.astype(bool) & ~pd.Series(EXPORT_GENES.isin(FIXED_GENES), index=EXPORT_GENES)
             & GS72[f'detection_{sp}'].ge(.10) & REFERENCE[sp][1].ge(1) & REFERENCE[sp][0].notna()) for sp in PAIRS}
FOLD_ALL = GS72.fold.astype(int)

ZW = {'P': {k: refit_coordinate(f'fold{k}') for k in range(3)},
      'D': {k: refit_coordinate(f'fold{k}', 'dpt_pseudotime') for k in range(3)},
      'PL': {k: z_pl for k in range(3)}}
FOLDS_W = [k for k in range(3) if all(ZW[arm][k] is not None for arm in ZW)]
READING.append(f'WSR uses folds {FOLDS_W} for every arm: a fold whose refit failed 72\'s orientation guard in any arm is '
               'dropped for all arms, so P, D and the placebo are scored on the same held-out genes.')
UW = np.logical_and.reduce([np.isfinite(ZW[arm][k]) for arm in ZW for k in FOLDS_W])
uw = np.flatnonzero(UW)
LIB_FULL = np.asarray(X_ALL.sum(axis=1)).ravel().astype(float)
eval_union = sorted(set(EVAL['mouse'].index[EVAL['mouse']]) | set(EVAL['human'].index[EVAL['human']]))
EX = log_normalize(X_ALL[uw][:, EXPORT_GENES.get_indexer(eval_union)], LIB_FULL[uw])
EX_INDEX = pd.Index(eval_union)
cov_w = np.column_stack([np.log(LIB_FULL[uw]), np.log(U.obs.n_spots.to_numpy(float)[uw]),
                         np.log(np.asarray((X_ALL[uw] > 0).sum(axis=1)).ravel().astype(float))])
spec_w, seg_w = SPEC_ALL[uw], pd.Series(SEG_ALL[uw]).map(SHORT_SEG).to_numpy()
anatomy_w = pd.Series(seg_w).map({'S1': 0, 'S2': 1, 'S3': 2}).to_numpy()


def wsr_payload():
    rng = np.random.default_rng(WSR_SEED)
    perms = {}
    for n in sorted(set(spec_w)):
        for s in ('S1', 'S2', 'S3'):
            size = int(((spec_w == n) & (seg_w == s)).sum())
            perms[(n, s)] = np.array([rng.permutation(size) for _ in range(WSR_PERMUTATIONS)])
    resid, columns = {}, {}
    for sp, names in PAIRS.items():
        for k in FOLDS_W:
            genes_k = [g for g in eval_union if EVAL[sp][g] and FOLD_ALL[g] == k]
            columns[(sp, k)] = genes_k
            cols = EX_INDEX.get_indexer(genes_k)
            for n in names:
                m = spec_w == n
                y = EX[m][:, cols].toarray()
                own = fit_segment_covariate_oracle(y, cov_w[m], anatomy_w[m], spec_w[m], degree=2)
                resid[(sp, k, n)] = y - predict_segment_covariate_oracle(own, cov_w[m], anatomy_w[m])
    rows = {key: [] for key in ('arm', 'species', 'segment', 'train', 'test', 'WSR', 'null mean', 'null max', 'p', 'genes')}
    for arm in ZW:
        for sp, names in PAIRS.items():
            for train, test in ((names[0], names[1]), (names[1], names[0])):
                tr, te = spec_w == train, spec_w == test
                obs, null = {s: [] for s in ('S1', 'S2', 'S3')}, {s: [] for s in ('S1', 'S2', 'S3')}
                for k in FOLDS_W:
                    z = ZW[arm][k][uw]
                    keep = np.array([arm != 'PL' or g not in PL_FEATURES for g in columns[(sp, k)]], bool)
                    res = wsr_correlations(resid[(sp, k, train)][:, keep], z[tr], seg_w[tr], resid[(sp, k, test)][:, keep],
                                           z[te], seg_w[te], {s: perms[(test, s)] for s in ('S1', 'S2', 'S3')}, df=WSR_DF)
                    for s, (o, nl) in res.items():
                        obs[s].append(o)
                        null[s].append(nl)
                for s in ('S1', 'S2', 'S3'):
                    o, nl = np.concatenate(obs[s]), np.concatenate(null[s], axis=1)
                    stat, null_stats = wsr_statistic(o), wsr_statistic(nl)
                    for key, value in (('arm', arm), ('species', sp), ('segment', s), ('train', train), ('test', test),
                                       ('WSR', stat), ('null mean', float(np.mean(null_stats))),
                                       ('null max', float(np.max(null_stats))),
                                       ('p', float((1 + np.sum(null_stats >= stat)) / (1 + len(null_stats)))),
                                       ('genes', int(len(o)))):
                        rows[key].append(value)
    return {k: np.asarray(v) for k, v in rows.items()}


WSR = pd.DataFrame(cached('wsr', wsr_payload, params={'permutations': WSR_PERMUTATIONS, 'seed': WSR_SEED, 'df': WSR_DF,
                                                      'folds': FOLDS_W},
                          inputs={'units': uw, 'eval': {sp: digest(EVAL[sp]) for sp in EVAL}, 'placebo': sorted(PL_FEATURES),
                                  'coordinates': {arm: [ZW[arm][k][uw] for k in FOLDS_W] for arm in ZW}}))
WSR.to_csv(tables / 'wsr.csv', index=False)
display(WSR.pivot_table(index=['species', 'segment'], columns=['arm', 'train'], values=['WSR', 'p']).round(4))

## 9 · R6 (part 2) · The comparators on the matched set M (separable, O1)

SCF13, DPT13, LCP and LRS-Z enter only through their cached positions on 72's matched set M (package PT units whose
notebook 70 match is an SCF13 PT structure); the package and the package DPT are scored on M too. Every arm uses the
package counts and labels, its own common support and α (the package's: 71's; the package DPT's: D's; the others:
their own under their species-split M_full), and its own construction genes for fold protection, plus the 15 panel
and anchor genes, which set the labels every arm uses. The information side is R4's (i), (ii) and (iv) with step
nulls (labels only: no (i-b) slopes on M). `RUN_COMPARATORS = False` drops this section and nothing else.

In [ ]:
M_ARMS, M_DESC, M_NULLS, M_OBSERVED, M_COMPARISON = {}, {}, {}, {}, {}
M_TABLE = None
if RUN_COMPARATORS:
    saved = ad.read_h5ad(results_root / 'minimal_pt_scfates' / 'cross_species_pt_scfates.h5ad', backed='r')
    saved_pt = saved.obs[['shared_pseudospace', 'dpt_pseudospace']].copy()
    saved.file.close()
    saved_ids = pd.Index(saved_pt.index.astype(str))
    lcp_table = pd.read_csv(results_root / 'pt_reconstruction_v3' / 'lcp_coordinate.csv')
    lrsz = pd.read_csv(results_root / 'pt_reconstruction_v3' / 'coordinates' / 'lrs_z.csv').set_index('structure_id')
    OLD = {'SCF13': saved_pt.shared_pseudospace.to_numpy(float), 'DPT13': saved_pt.dpt_pseudospace.to_numpy(float),
           'LCP': pd.Series(lcp_table.position.to_numpy(float), index=lcp_table.structure_id.astype(str)).reindex(saved_ids).to_numpy(),
           'LRS-Z': lrsz.position.reindex(saved_ids).to_numpy(float)}
    nb36_payload = sorted((results_root / 'pt_reconstruction_v2' / 'nb36' / 'stage_cache').glob('arm_scf13_full__*.npz'))
    scf_construction = set(np.load(nb36_payload[0], allow_pickle=True)['construction'].astype(str))
    CONSTRUCTION = {'package': BASE_FEATURES, 'package DPT': BASE_FEATURES, 'SCF13': scf_construction,
                    'DPT13': scf_construction,
                    'LCP': set(pd.read_csv(results_root / 'pt_reconstruction_v3' / 'nb51' / 'landmark_panel.csv').gene.astype(str)),
                    'LRS-Z': set((results_root / 'pt_reconstruction_v3' / 'coordinates' / 'lrs_z_construction_genes.txt').read_text().split())}
    old_ids = U.obs.old_structure_id.astype(object)
    M_ROWS = np.flatnonzero(old_ids.isin(saved_ids).to_numpy())
    M_INDEX = saved_ids.get_indexer(old_ids.iloc[M_ROWS].astype(str))
    unit_set_m = pd.read_csv(nb72 / 'tables' / 'unit_set_M.csv')
    assert list(unit_set_m.unit.astype(str)) == list(IDS[M_ROWS]), 'M differs from notebook 72\'s unit_set_M.csv'
    for arm in ('package', 'package DPT', *OLD):
        raw = np.full(len(IDS), np.nan)
        own = {'package': RAW_ALL[M_ROWS], 'package DPT': DPT_ALL[M_ROWS]}
        raw[M_ROWS] = own[arm] if arm in own else OLD[arm][M_INDEX]
        M_ARMS[arm] = view(raw, f'M {arm}')
        FOLD_PROTECTED[f'M {arm}'] = ~pd.Index(GENES).isin(sorted(CONSTRUCTION[arm] | set(FIXED_GENES)))
        if arm == 'package':
            M_ARMS[arm]['alpha'] = ALPHA71
        elif arm == 'package DPT':
            M_ARMS[arm]['alpha'] = V['D']['alpha']
        else:
            vm = M_ARMS[arm]
            dm, wm = contrast_designs(vm['position'], vm['human'].astype(float), vm['specimen'], vm['knots'])
            a = cached(f'alpha_M_{arm}', lambda vm=vm, dm=dm, wm=wm: {k: x for k, x in nb_dispersion(
                vm['counts'], dm['full'], wm, np.log(vm['library']), n_jobs=N_JOBS).items() if k in ('alpha', 'converged')},
                params={'basis_df': BASIS_DF, 'arm': arm}, inputs=vm['key'])
            vm['alpha'] = {'alpha': a['alpha'], 'converged': a['converged'].astype(bool)}
    for arm, vm in M_ARMS.items():
        M_DESC[arm] = cached(f'descriptors_M_{arm}', lambda vm=vm: descriptor_payload(vm, vm['counts'], vm['library'],
                                                                                       coord_slopes=False),
                             params={'rules': [MIN_AMPLITUDE, MIN_ABS_RHO, MIN_RANGE, MIN_PROMINENCE, INNER], 'arm': arm},
                             inputs={**vm['key'], 'alpha': vm['alpha']['alpha'], 'coord_segment': vm['coord_segment']})

        def m_null(vm=vm, arm=arm, null_arm='A', rep=1):
            blocks = np.array([f'{a}|{b}' for a, b in zip(vm['specimen'], vm['coord_segment'] if null_arm == 'A' else vm['segment'])])
            index = blockwise_permutation(blocks, np.random.default_rng(NULL_SEEDS_58[null_arm] + rep))
            Dn = descriptor_payload(vm, vm['counts'][index], vm['library'][index], coord_slopes=False)
            return {k: np.array(x, float) for k, x in descriptor_metrics(Dn, FOLD_PROTECTED[f'M {arm}'], 'ln').items()}

        for null_arm in 'AB':
            for rep in range(1, N_REPLICATES + 1):
                M_NULLS[(arm, null_arm, rep)] = cached(
                    f'null58_M_{arm}_{null_arm}_{rep}', lambda m_null=m_null, null_arm=null_arm, rep=rep: m_null(null_arm=null_arm, rep=rep),
                    params={'arm': null_arm, 'rep': rep, 'seed': NULL_SEEDS_58[null_arm], 'coordinate': arm,
                            'rules': [MIN_AMPLITUDE, MIN_ABS_RHO, MIN_RANGE, MIN_PROMINENCE, INNER]},
                    inputs={**vm['key'], 'alpha': vm['alpha']['alpha'], 'coord_segment': vm['coord_segment'],
                            'protected': digest(FOLD_PROTECTED[f'M {arm}'])})
        M_OBSERVED[arm] = pd.Series(descriptor_metrics(M_DESC[arm], FOLD_PROTECTED[f'M {arm}'], 'ln', boot=True))
        frame = pd.DataFrame([{**{k: float(x) for k, x in M_NULLS[(arm, a, r)].items()}, 'arm': a}
                              for a in 'AB' for r in range(1, N_REPLICATES + 1)])
        cmp = pd.DataFrame({'observed': M_OBSERVED[arm].reindex(frame.columns.drop('arm')),
                            'arm A max': frame[frame.arm.eq('A')].drop(columns='arm').max(),
                            'arm B max': frame[frame.arm.eq('B')].drop(columns='arm').max()})
        cmp['exceeds every null replicate'] = cmp.observed > cmp[['arm A max', 'arm B max']].max(axis=1)
        M_COMPARISON[arm] = cmp
    M_TABLE = pd.concat({arm: t for arm, t in M_COMPARISON.items()}, names=['arm', 'metric'])
    M_TABLE.to_csv(tables / 'comparators_information_M.csv')
    display(pd.DataFrame({arm: M_OBSERVED[arm] for arm in M_ARMS}).round(3))
else:
    PENDING.append('M comparators: switched off (RUN_COMPARATORS = False)')

## 10 · R5 · Onsets and slopes in external adult male mice (notebook 59, NB)

- **External coordinate (unchanged; an embedding, not a gene model, 72's O3):** per Census donor (adult male mouse PT
  nuclei, 3–6 and 12 months), log1p(count / total × 10⁴), 2,000 seurat-flavour HVGs that exclude every gene of R4's
  (iv) primary sets on any coordinate and the six orientation markers, PCA 30, 30 neighbours, diffusion pseudotime
  rooted at the highest early-marker score, scaled to [0, 1]; a donor is used when S1 < S2 < S3.
- **External curves and slopes: NB** on raw counts, offset log(nucleus total without mt), α estimated per donor under
  that donor's spline design; onsets, r and slopes by R4's rules (natural-log scale; log1p as the sensitivity).
- **Primary (59's):** partial Spearman of our mouse onset (mean of the two mice, R4's primary set) with the external
  onset (median over ≥ 3 donors monotone in our direction), given our r; 2,000 donor bootstraps (seed 59). Claim if the
  P value is ≥ 0.2 with the interval above 0; abandon framing F2(a) if it is < 0.1 here and R4's (iv) is < 0.1.
- **Secondary (pre-specified now):** within-segment slope agreement with the external donor median, raw and given the
  external S2 − S1 and S3 − S2 steps.

In [ ]:
import scanpy as sc

census_file = data_root / 'external' / 'census_pt_cells' / 'mouse_pt_cells.h5ad'
ADULT_STAGES = ('3-6 month-old mature stage', '12-month-old stage')
SEGMENT_MAP = {'epithelial cell of proximal tubule segment 1': 'PT-S1',
               'epithelial cell of proximal tubule segment 2': 'PT-S2',
               'epithelial cell of proximal tubule segment 3': 'PT-S3'}
EARLY, LATE = ('Slc5a2', 'Slc5a12', 'Gatm'), ('Slc22a7', 'Slc7a13', 'Cyp7b1')
N_HVG, N_PCS, N_NEIGHBORS, SEED59, MIN_DONORS, CLAIM59, ABANDON59 = 2000, 30, 30, 59, 3, .2, .1
OURS = {code: IV_GENES[code] for code in ('P', 'D', 'E') if code in IV_GENES}
PROTECTED = sorted(set().union(*[set(t.index) for t in OURS.values()]) | set(EARLY) | set(LATE))
census = ad.read_h5ad(census_file)
obs_c = census.obs
keep_c = (obs_c.sex.astype(str).eq('male') & obs_c.development_stage.astype(str).isin(ADULT_STAGES)
          & obs_c.cell_type.astype(str).isin(SEGMENT_MAP)).to_numpy()
census = census[keep_c].copy()
census.obs['segment'] = census.obs.cell_type.astype(str).map(SEGMENT_MAP)
DONORS = sorted(census.obs.donor_id.astype(str).unique())
CENSUS_GENES = pd.Index(census.var_names.astype(str))
NEEDED = [g for g in GENES if g in set(CENSUS_GENES) and (g in PROTECTED or FOLD_PROTECTED['P'][GENES.index(g)])]
CENSUS_MT = CENSUS_GENES.str.startswith('mt-')
display(census.obs.groupby(['donor_id', 'development_stage', 'segment'], observed=True).size().unstack())


def donor_descriptors(donor):
    sub = census[census.obs.donor_id.astype(str).eq(donor).to_numpy()]
    counts = sparse.csr_matrix(sub.X, dtype=float)
    total = np.asarray(counts.sum(axis=1)).ravel()
    lognorm = counts.multiply((1e4 / np.maximum(total, 1))[:, None]).tocsr()
    lognorm.data = np.log1p(lognorm.data)
    work = ad.AnnData(X=lognorm, obs=sub.obs[['segment']].copy(), var=pd.DataFrame(index=CENSUS_GENES))
    candidates = ~work.var_names.isin(PROTECTED)
    hv = sc.pp.highly_variable_genes(work[:, candidates].copy(), flavor='seurat', n_top_genes=N_HVG, inplace=False)
    hvg = work.var_names[candidates][hv.highly_variable.to_numpy()]
    emb = work[:, hvg].copy()
    sc.pp.pca(emb, n_comps=N_PCS, random_state=SEED59)
    sc.pp.neighbors(emb, n_neighbors=N_NEIGHBORS, use_rep='X_pca', random_state=SEED59)
    sc.tl.diffmap(emb, random_state=SEED59)
    markers = work[:, list(EARLY + LATE)].X.toarray()
    zm = (markers - markers.mean(axis=0)) / np.maximum(markers.std(axis=0), 1e-6)
    emb.uns['iroot'] = int(np.argmax(zm[:, :3].mean(axis=1) - zm[:, 3:].mean(axis=1)))
    sc.tl.dpt(emb)
    t = emb.obs.dpt_pseudotime.to_numpy(float)
    finite = np.isfinite(t)
    t = (t - t[finite].min()) / np.ptp(t[finite])
    seg = work.obs.segment.astype(str).to_numpy()
    medians = pd.Series(t[finite]).groupby(seg[finite]).median()
    ordered = bool(medians['PT-S1'] < medians['PT-S2'] < medians['PT-S3'])
    lo, hi = np.quantile(t[finite], [.01, .99])
    rows = finite & (t >= lo) & (t <= hi)
    out = {'ordered': np.array(ordered), 'medians': medians.reindex(list(SEGMENTS)).to_numpy(float),
           'n_nuclei': np.array(int(rows.sum())), 'hvg_overlap_protected': np.array(len(set(hvg) & set(PROTECTED)))}
    if not ordered:
        return out
    grid = np.linspace(lo, hi, 61)
    position, specimen = t[rows], np.repeat(donor, rows.sum())
    y = counts[rows][:, CENSUS_GENES.get_indexer(NEEDED)]
    library = np.asarray(counts[rows][:, ~CENSUS_MT].sum(axis=1)).ravel()
    knots = gam_internal_knots(position, basis_df=BASIS_DF)
    dispersion = nb_dispersion(y, make_gam_design(position, knots), np.ones(len(position)), np.log(library), n_jobs=N_JOBS)
    curve = nb_specimen_curves(y, library, position, specimen, grid, knots, dispersion, n_jobs=N_JOBS)[donor]
    transitions = logistic_transitions(position, seg[rows], specimen).set_index('transition').position
    out['transitions'] = transitions.reindex(['S1→S2', 'S2→S3']).to_numpy(float)
    if not grid[0] < out['transitions'][0] < out['transitions'][1] < grid[-1]:
        out['ordered'] = np.array(False)
        return out
    units = segment_units(grid, *out['transitions'], grid[0], grid[-1])
    means = pseudobulk_log_rates(y, library, specimen, seg[rows])[donor]
    for scale in ('ln', 'log1p'):
        onsets = monotone_onsets(scaled_curve(curve, scale), units, min_amplitude=MIN_AMPLITUDE, min_abs_rho=MIN_ABS_RHO)
        out[f'{scale}|onset'] = onsets.onset.to_numpy(float)
        out[f'{scale}|direction'] = onsets.direction.to_numpy(float)
        out[f'{scale}|monotone'] = onsets.monotone.to_numpy(float)
        out[f'{scale}|r'] = switch_fraction(scaled_means(means, scale), MIN_RANGE)
    out['means'] = means
    out['slope'] = nb_segment_slopes(y, library, position, specimen, seg[rows], dispersion, n_jobs=N_JOBS)[donor].to_numpy(float)
    return out


EXTERNAL_DESC = {d: cached(f'external_donor_{d}', lambda d=d: donor_descriptors(d),
                           params={'hvg': N_HVG, 'pcs': N_PCS, 'neighbors': N_NEIGHBORS, 'seed': SEED59,
                                   'rules': [MIN_AMPLITUDE, MIN_ABS_RHO, MIN_RANGE], 'protected': PROTECTED, 'genes': NEEDED},
                           inputs={'file': [census_file.name, census_file.stat().st_size]})
                 for d in DONORS}
DONOR_TABLE = pd.DataFrame({d: {'nuclei in range': int(v['n_nuclei']), 'S1 < S2 < S3': bool(v['ordered']),
                                'median S1 / S2 / S3': ' / '.join(f'{x:.2f}' for x in v['medians']),
                                'protected genes among HVGs (must be 0)': int(v['hvg_overlap_protected'])}
                            for d, v in EXTERNAL_DESC.items()}).T
assert DONOR_TABLE['protected genes among HVGs (must be 0)'].sum() == 0
DONOR_TABLE.to_csv(tables / 'external_donors.csv')
USED = [d for d, v in EXTERNAL_DESC.items() if bool(v['ordered'])]
display(DONOR_TABLE)
needed_index = pd.Index(NEEDED)
rng59 = np.random.default_rng(SEED59)
BOOT_DONORS = [list(rng59.choice(USED, len(USED), replace=True)) for _ in range(N_BOOT)]


def external_onsets(direction, donors, scale):
    genes_ = direction.index
    idx = needed_index.get_indexer(genes_)
    on = np.column_stack([EXTERNAL_DESC[d][f'{scale}|onset'][idx] for d in donors])
    dr = np.column_stack([EXTERNAL_DESC[d][f'{scale}|direction'][idx] for d in donors])
    mo = np.column_stack([EXTERNAL_DESC[d][f'{scale}|monotone'][idx] for d in donors]).astype(bool)
    valid = mo & (dr == direction.to_numpy()[:, None])
    values = np.where(valid, on, np.nan)
    count = valid.sum(axis=1)
    with np.errstate(all='ignore'):
        med = np.array([np.median(v[np.isfinite(v)]) if c >= MIN_DONORS else np.nan for v, c in zip(values, count)])
    return pd.Series(med, index=genes_)


EXT_ROWS = []
for scale in ('ln', 'log1p'):
    for code, table in OURS.items():
        if scale == 'log1p':     # the sensitivity uses our own log1p onsets on the same coordinate
            _, table = ordering_test(onset_table(DESC[code], MICE, 'log1p'), 1, keep=FOLD_PROTECTED[code])
        t = table[table.index.isin(needed_index)].copy()
        our_onset, our_r = (t.onset_a + t.onset_b) / 2, (t.r_a + t.r_b) / 2
        direction = pd.Series(t.dir_a.to_numpy(), index=t.index)
        ext = external_onsets(direction, USED, scale)
        m = ext.notna() & our_r.notna()
        frame = pd.DataFrame({'our onset': our_onset, 'our r': our_r, 'external onset': ext})[m]
        if scale == 'ln':
            frame.to_csv(tables / f'external_onsets_{code}.csv')
        row = {'scale': scale, 'coordinate': code, 'genes in our set': len(t), 'genes with external onset': int(m.sum())}
        if m.sum() >= 20:
            est = partial_spearman(frame['our onset'], frame['external onset'], frame[['our r']].to_numpy())
            draws = []
            for chosen in BOOT_DONORS:
                e = external_onsets(direction[m], chosen, scale)
                ok = e.notna()
                draws.append(partial_spearman(frame['our onset'][ok], e[ok], frame[['our r']][ok].to_numpy())
                             if ok.sum() >= 20 else np.nan)
            row.update({'partial Spearman (external | our r)': est, '95% low (donor bootstrap)': float(np.nanquantile(draws, .025)),
                        '95% high (donor bootstrap)': float(np.nanquantile(draws, .975)),
                        'Spearman our onset vs external onset': spearmanr(frame['our onset'], frame['external onset']).statistic})
        EXT_ROWS.append(row)
EXTERNAL_ONSETS = pd.DataFrame(EXT_ROWS)
EXTERNAL_ONSETS.to_csv(tables / 'external_onsets.csv', index=False)
p_ext = EXTERNAL_ONSETS[EXTERNAL_ONSETS.scale.eq('ln') & EXTERNAL_ONSETS.coordinate.eq('P')]
ext_value = float(p_ext['partial Spearman (external | our r)'].iloc[0]) if 'partial Spearman (external | our r)' in p_ext and len(p_ext) else np.nan
ext_low = float(p_ext['95% low (donor bootstrap)'].iloc[0]) if '95% low (donor bootstrap)' in p_ext and len(p_ext) else np.nan
if ext_value >= CLAIM59 and ext_low > 0:
    DECISION_R5 = 'claim: the within-S2 onset order replicates in external mice beyond segment means'
elif ext_value < ABANDON59 and iv_value < ABANDON59:
    DECISION_R5 = 'abandon framing F2(a)'
else:
    DECISION_R5 = 'not replicated externally at the pre-specified level'
display(EXTERNAL_ONSETS.round(3))
print('R5 decision:', DECISION_R5)

ext_slopes = {k: pd.DataFrame({d: EXTERNAL_DESC[d]['slope'][:, k] for d in USED}, index=needed_index) for k in range(3)}
ext_steps = {'S2-S1': pd.DataFrame({d: EXTERNAL_DESC[d]['means'][:, 1] - EXTERNAL_DESC[d]['means'][:, 0] for d in USED},
                                   index=needed_index).median(axis=1),
             'S3-S2': pd.DataFrame({d: EXTERNAL_DESC[d]['means'][:, 2] - EXTERNAL_DESC[d]['means'][:, 1] for d in USED},
                                   index=needed_index).median(axis=1)}
SLOPE_ROWS = []
for code in ('P', 'D', 'E'):
    if code not in DESC:
        continue
    protected_genes = pd.Index(GENES)[FOLD_PROTECTED[code]].intersection(needed_index)
    for k, seg in enumerate(('S1', 'S2', 'S3')):
        ext_med = ext_slopes[k].reindex(protected_genes).median(axis=1)
        for kind, key in (('label', 'slope'), ('coordinate', 'slope_c')):
            if f'{MICE[0]}|{key}' not in DESC[code]:
                continue
            ours_t = pd.Series(np.nanmean([DESC[code][f'{n}|{key}'][:, k] for n in MICE], axis=0), index=GENES).reindex(protected_genes)
            ok = ours_t.notna() & ext_med.notna()
            steps = pd.DataFrame({k_: v_.reindex(protected_genes) for k_, v_ in ext_steps.items()})
            okp = ok & steps.notna().all(axis=1)
            draws = []
            for chosen in BOOT_DONORS[:500]:
                e = ext_slopes[k].reindex(protected_genes)[chosen].median(axis=1)
                k_ = ok & e.notna()
                draws.append(spearmanr(ours_t[k_], e[k_]).statistic)
            SLOPE_ROWS.append({'coordinate': code, 'segment': seg, 'our slopes inside': kind, 'genes': int(ok.sum()),
                               'Spearman with external slopes': spearmanr(ours_t[ok], ext_med[ok]).statistic,
                               'partial Spearman given external segment steps': partial_spearman(ours_t[okp], ext_med[okp],
                                                                                                 steps[okp].to_numpy()),
                               '95% low (donor bootstrap)': float(np.nanquantile(draws, .025)),
                               '95% high (donor bootstrap)': float(np.nanquantile(draws, .975))})
EXTERNAL_SLOPES = pd.DataFrame(SLOPE_ROWS)
EXTERNAL_SLOPES.to_csv(tables / 'external_slopes.csv', index=False)
display(EXTERNAL_SLOPES.round(3))
del census

## 11 · Sections that read notebook 75 (G3)

- **R2 (part 2) · continuous against discrete calls.** A = notebook 71's calls (positional F-test + cameraPR);
  S = notebook 75's segment F-test calls (its twin with the package S1/S2/S3 labels in place of the spline). Counts in
  sets and in 71's redundancy clusters; for each set in A∖S: S's FDR ("near miss" if ≤ 0.25, descriptive), whether
  test B calls it, whether 75's segment ORA calls it, and probe sensitivity (cameraPR on 71's z restricted to the
  balanced-probe genes gives FDR > 0.10, or the set's mean-z contrast falls below 0.7 of the full one; 37's rule).
- **Fig 3e categories of A:** also called by S; continuous-only; beyond coordinate steps (test B); sign change (the
  members' mean Δ reaches ≥ 0.25 · max|Δ| on both signs, 61's E4 rule); location (the segment of max |Δ|).
- **R3 · worked examples** (61's E1–E5 with NB statistics) and **R4 (iii) · boundary shifts** (programs from 75).

When 75's tables are absent, these sections are listed as pending.

In [ ]:
def read_blocks(path):
    """A table in 71/75's layout (column blocks per partition); returns the condition block (or the table)."""
    table = pd.read_csv(path, header=[0, 1], index_col=0)
    return table['condition'] if 'condition' in table.columns.get_level_values(0) else table.droplevel(0, axis=1)


def evidence_of(frame):
    """-log10 of a set table's q (fdr, q, or the smallest q over an ORA family's lists), for ranking."""
    family = [c for c in frame.columns if str(c).startswith('q_')]
    if 'fdr' not in frame and 'q' not in frame and family:
        q = frame[family].apply(pd.to_numeric, errors='coerce').min(axis=1).fillna(1).to_numpy(float)
        return -np.log10(np.clip(q, 1e-300, 1))
    for column in ('fdr', 'q', 'best_p', 'p'):
        if column in frame:
            return -np.log10(np.clip(pd.to_numeric(frame[column], errors='coerce').fillna(1).to_numpy(float), 1e-300, 1))
    raise KeyError(f'no q or p column among {list(frame.columns)[:10]}')


T75 = nb75 / 'tables'
READY_75 = all((T75 / f).is_file() for f in ('gene_statistics_segment_human_vs_mouse.csv', 'pathway_tests_segment_human_vs_mouse.csv'))
S_CALLS, S_FDR, S_Z, S_PASS = set(), pd.Series(dtype=float), pd.Series(dtype=float), None
RULE_CALLS, RULE_EVIDENCE = {}, {}
CONT_VS_SEG, CONT_ADDS, SEG_ADDS, CATEGORIES, FIG3E, seg_genes = None, None, None, None, None, None
EXAMPLE_TABLE = None
names = pd.Series(list(SETS), index=list(SETS)).str.split('::').str[1].str.replace(r' R-HSA-\d+$', '', regex=True)
test_b = R1_SETS[('P', 'B', 'species')].set_index('pathway_id')
B_CALLS = set(test_b.index[test_b.call.astype(bool)])
good71 = M71['stat|converged'].astype(bool) & ~M71['stat|separated'].astype(bool)
gene_index = pd.Index(GENES)
t_mean = {k: float(np.mean(DESC['P'][k])) for k in ('t12', 't23')}


def pathway_delta(p):
    """A set's species difference along P: the mean of its converged, non-separated members' 71 Δ (natural log)."""
    idx = gene_index.get_indexer(SETS[p])
    idx = idx[good71[idx]]
    return M71['delta'][idx].mean(axis=0)


if READY_75:
    seg_sets = pd.read_csv(T75 / 'pathway_tests_segment_human_vs_mouse.csv', header=[0, 1], index_col=0)
    seg_genes = pd.read_csv(T75 / 'gene_statistics_segment_human_vs_mouse.csv', header=[0, 1], index_col=0)
    assert set(seg_sets.index) == set(SETS) and list(seg_genes.index.astype(str)) == GENES, 'G3: 75 is not on 71\'s universe and sets'
    blocks = list(dict.fromkeys(seg_sets.columns.get_level_values(0)))
    cond = seg_sets['condition']
    S_CALLS = set(cond.index[cond['call'].astype(bool)])
    S_FDR = pd.to_numeric(cond['fdr'], errors='coerce')
    S_Z = pd.to_numeric(seg_genes['condition']['z'], errors='coerce')
    S_PASS = swap_rule(len(S_CALLS), [int(seg_sets[b]['call'].astype(bool).sum()) for b in blocks if b != 'condition'],
                       len(cond), np.concatenate([pd.to_numeric(seg_sets[b]['p_one_sided']).to_numpy() for b in blocks
                                                  if b != 'condition']))
    for key, stem in (('A1·ORAc', 'deseq2_wholept_ora'), ('A3·ORAc', 'deseq2_segments_ora'), ('A3·GSEA', 'deseq2_segments_gsea')):
        path = T75 / f'pathway_tests_{stem}_human_vs_mouse.csv'
        if path.is_file():
            frame = read_blocks(path)
            RULE_CALLS[key] = set(frame.index[frame['call'].astype(bool)])
            RULE_EVIDENCE[key] = pd.Series(evidence_of(frame), index=frame.index)
        else:
            PENDING.append(f'R2/R3: 75\'s {stem} table is absent; rules that use {key} are skipped')
    RULE_EVIDENCE['A6·NB'] = pd.Series(evidence_of(CALLS71), index=CALLS71.index)
    RULE_EVIDENCE['A4·NB'] = pd.Series(-np.log10(np.clip(S_FDR.to_numpy(float), 1e-300, 1)), index=S_FDR.index)

    # Probe sensitivity of the continuous calls (37's rule on 71's z, without refitting)
    probes = pd.read_csv(results_root / 'pt_literature_deep' / 'probe_counts.csv').drop_duplicates('mouse_symbol').set_index('mouse_symbol')
    balanced = set(probes.index[probes.probe_balance.eq('balanced')]) & set(GENES)
    z71 = pd.Series(STAT71['z'].to_numpy(float), index=GENES)
    restricted_sets = {p: [g for g in m if g in balanced] for p, m in SETS.items()}
    restricted_sets = {p: m for p, m in restricted_sets.items() if 10 <= len(m) <= 300}
    probe_test = camera(z71[sorted(balanced)], restricted_sets).set_index('pathway_id')

    def contrast(z, members):
        inside = z.index.isin(members)
        return z[inside].mean() - z[~inside].mean()

    def probe_sensitive(p):
        if p not in probe_test.index:
            return True
        ratio = contrast(z71[sorted(balanced)], restricted_sets[p]) / contrast(z71, SETS[p])
        return bool(probe_test.loc[p, 'fdr'] > PROBE_Q or ratio < PROBE_EFFECT)

    A, Sc = C71, S_CALLS
    cl = CLUSTERS['hvm']
    CONT_VS_SEG = pd.DataFrame([{'unit': unit, 'both': n(A & Sc), 'continuous only (A∖S)': n(A - Sc), 'segment only (S∖A)': n(Sc - A)}
                                for unit, n in (('sets', len), ('redundancy clusters', lambda s: cl.reindex(sorted(s)).nunique()))])
    CONT_VS_SEG.to_csv(tables / 'continuous_vs_segment.csv', index=False)
    CONT_ADDS = pd.DataFrame({'pathway': names[sorted(A - Sc)], 'redundancy cluster': cl.reindex(sorted(A - Sc)),
                              '71 FDR': CALLS71.fdr.reindex(sorted(A - Sc)), 'segment F-test FDR (75)': S_FDR.reindex(sorted(A - Sc)),
                              'near miss (segment FDR ≤ 0.25)': S_FDR.reindex(sorted(A - Sc)).le(NEAR_MISS),
                              'test B calls it': [p in B_CALLS for p in sorted(A - Sc)],
                              "75's segment ORA calls it": [p in RULE_CALLS.get('A3·ORAc', set()) for p in sorted(A - Sc)],
                              'probe-sensitive': [probe_sensitive(p) for p in sorted(A - Sc)]}).sort_values('segment F-test FDR (75)')
    CONT_ADDS.to_csv(tables / 'continuous_adds.csv')
    SEG_ADDS = pd.DataFrame({'pathway': names[sorted(Sc - A)], 'segment F-test FDR (75)': S_FDR.reindex(sorted(Sc - A)),
                             '71 FDR': CALLS71.fdr.reindex(sorted(Sc - A))})
    SEG_ADDS.to_csv(tables / 'segment_adds.csv')
    display(CONT_VS_SEG)
    display(CONT_ADDS.round(3))
    display(SEG_ADDS.round(3))
    print("75's segment F-test pass rule:", S_PASS)

    # Fig 3e categories (pathway Δ from 71's cached Δ, converged non-separated members)
    cat_rows = []
    for p in sorted(A):
        d = pathway_delta(p)
        top = np.abs(d).max()
        peak = P['grid'][int(np.argmax(np.abs(d)))]
        cat_rows.append({'pathway_id': p, 'pathway': names[p], 'redundancy cluster': cl.get(p),
                         'also called by S': p in Sc, 'continuous only': p not in Sc, 'beyond coordinate steps (test B)': p in B_CALLS,
                         'sign change along the PT': bool((d >= REVERSAL * top).any() and (d <= -REVERSAL * top).any()),
                         'location of max |Δ|': 'S1' if peak < t_mean['t12'] else 'S2' if peak < t_mean['t23'] else 'S3',
                         'direction at max |Δ|': 'human higher' if d[int(np.argmax(np.abs(d)))] > 0 else 'mouse higher'})
    CATEGORIES = pd.DataFrame(cat_rows).set_index('pathway_id')
    CATEGORIES.to_csv(tables / 'fig3e_categories.csv')
    FIG3E = pd.DataFrame({'sets': [int(CATEGORIES[c].sum()) for c in ('also called by S', 'continuous only',
                                                                     'beyond coordinate steps (test B)', 'sign change along the PT')],
                          'redundancy clusters': [CATEGORIES.loc[CATEGORIES[c], 'redundancy cluster'].nunique()
                                                  for c in ('also called by S', 'continuous only',
                                                            'beyond coordinate steps (test B)', 'sign change along the PT')]},
                         index=['also called by S', 'continuous only', 'beyond coordinate steps (test B)', 'sign change along the PT'])
    display(FIG3E)
    display(CATEGORIES.groupby(['location of max |Δ|', 'direction at max |Δ|']).size().unstack(fill_value=0))
else:
    PENDING.append("R2 comparisons, Fig 3e, R3 worked examples and R4 (iii): notebook 75's segment F-test tables are absent")
    print('Pending: notebook 75 has not written its segment F-test tables yet.')

### R3 · Worked examples: one pathway's lead gene, three ways (Figure 3a)

61's rules with NB statistics: continuous calls are 71's, step calls 75's segment F-test, test B is R1's on P, and
whole-PT ORA, segment ORA and segment GSEA are 75's DESeq2 rows.
- **E1** all agree (whole-PT ORA, segment ORA, segment F-test, continuous): the highest minimum percentile rank.
- **E2** segment F-test FDR > 0.05, no segment GSEA or ORA call, but test B calls it: the largest test-B set z.
- **E3** a segment-ORA call without a continuous call: the strongest segment-ORA evidence.
- **E4** the members' mean Δ reverses (≥ 0.25 · max|Δ| on both signs): the strongest continuous evidence.
- **E5** called by both the continuous and the segment F-test, not by test B: the strongest segment evidence.

**Lead gene:** the member with the largest gene z of the rule's statistic (71's z for E1 and E4, and in E4 the member's
own Δ must reverse; test-B z for E2; segment-F z for E5 and E3). Each lead gene is drawn three ways: whole-PT specimen
pseudobulk, specimen × segment pseudobulk, and 71's NB species curves with specimen × 12-bin pseudobulk points. Display
only: no test.

In [ ]:
EXAMPLES = {}
if READY_75 and all(k in RULE_CALLS for k in ('A1·ORAc', 'A3·ORAc', 'A3·GSEA')):
    z71 = pd.Series(STAT71['z'].to_numpy(float), index=GENES)
    z_b = pd.Series(R1_GENES[('P', 'B', 'species')].z.to_numpy(float), index=GENES)
    good_genes = pd.Series(good71, index=GENES)

    def reverses(d):
        top = np.abs(d).max()
        return bool((d >= REVERSAL * top).any() and (d <= -REVERSAL * top).any())

    ranks = pd.DataFrame({k: RULE_EVIDENCE[k] for k in ('A1·ORAc', 'A3·ORAc', 'A4·NB', 'A6·NB')}).rank(pct=True)
    agree = sorted(RULE_CALLS['A1·ORAc'] & RULE_CALLS['A3·ORAc'] & S_CALLS & C71)
    flat = [p for p in sorted(C71) if S_FDR.get(p, 1) > ALPHA and p not in RULE_CALLS['A3·GSEA']
            and p not in RULE_CALLS['A3·ORAc'] and p in B_CALLS]
    only_ora = sorted(RULE_CALLS['A3·ORAc'] - C71)
    reversing = [p for p in sorted(C71) if reverses(pathway_delta(p))]
    steplike = [p for p in sorted(C71 & S_CALLS) if p not in B_CALLS]
    picks = {'E1 · all methods agree': ranks.loc[agree].min(axis=1).sort_values(ascending=False).index[:1] if agree else [],
             'E2 · segment steps flat, curves structured': pd.Series(set_z(test_b.p_one_sided), index=test_b.index)[flat]
             .sort_values(ascending=False).index[:1] if flat else [],
             'E3 · discrete call, no continuous call': RULE_EVIDENCE['A3·ORAc'][only_ora].sort_values(ascending=False).index[:1]
             if only_ora else [],
             'E4 · direction depends on position': RULE_EVIDENCE['A6·NB'][reversing].sort_values(ascending=False).index[:1]
             if reversing else [],
             'E5 · step-like: continuous adds nothing': RULE_EVIDENCE['A4·NB'][steplike].sort_values(ascending=False).index[:1]
             if steplike else []}
    counts_rule = {'E1': len(agree), 'E2': len(flat), 'E3': len(only_ora), 'E4': len(reversing), 'E5': len(steplike)}
    lead_stat = {'E1': z71, 'E2': z_b, 'E3': S_Z, 'E4': z71, 'E5': S_Z}
    for rule, picked in picks.items():
        if not len(picked):
            print(f'{rule}: no pathway meets the rule')
            continue
        p = picked[0]
        members = [g for g in SETS[p] if good_genes.get(g, False)]
        if rule.startswith('E4'):
            members = [g for g in members if reverses(M71['delta'][GENES.index(g)])] or members
        lead = lead_stat[rule[:2]].reindex(members).idxmax()
        EXAMPLES[rule] = (p, lead)
    if 'A3·ORAc' in RULE_CALLS:
        READING.append('R3: E3\'s lead gene uses the segment F-test z (75\'s per-gene table), the gene statistic shared by every '
                       'segment rule; the protocol named |DESeq2 Wald|, which 75 stores per scope.')
    EXAMPLE_TABLE = pd.DataFrame([{'example': rule, 'pathway': names[p], 'pathway_id': p, 'lead gene': g,
                                   'pathways meeting the rule': counts_rule[rule[:2]],
                                   'continuous (71) FDR': CALLS71.fdr.get(p, np.nan), 'segment F-test FDR': S_FDR.get(p, np.nan),
                                   'test B called': p in B_CALLS, 'segment ORA called': p in RULE_CALLS['A3·ORAc'],
                                   'whole-PT ORA called': p in RULE_CALLS['A1·ORAc'],
                                   'lead gene 71 q': float(STAT71['q'].get(g, np.nan)),
                                   'lead gene segment-F q': float(pd.to_numeric(seg_genes['condition']['q'], errors='coerce').get(g, np.nan)),
                                   'lead gene test-B q': float(R1_GENES[('P', 'B', 'species')].q.get(g, np.nan))}
                                  for rule, (p, g) in EXAMPLES.items()])
    EXAMPLE_TABLE.to_csv(tables / 'worked_examples.csv', index=False)
    display(EXAMPLE_TABLE.round(4))
elif READY_75:
    PENDING.append('R3: one of 75\'s DESeq2 rule tables is absent')

if EXAMPLES:
    MOUSE_C, HUMAN_C, MUTED = '#2a78d6', '#eb6834', '#8a8984'
    MARKER = {MICE[0]: 'o', MICE[1]: 's', HUMANS[0]: 'o', HUMANS[1]: 's'}
    LOG10K = np.log(1e4)
    bins12 = np.clip((P['position'] * EXAMPLE_BINS).astype(int), 0, EXAMPLE_BINS - 1)
    fig, axes = plt.subplots(len(EXAMPLES), 3, figsize=(7.2, 1.6 * len(EXAMPLES)), gridspec_kw={'width_ratios': [.55, 1, 1.9]},
                             squeeze=False)
    limits_c = {}
    for r, (rule, (p, g)) in enumerate(EXAMPLES.items()):
        j = GENES.index(g)
        y = np.asarray(P['counts'][:, j].todense()).ravel()
        ax_w, ax_s, ax_c = axes[r]
        for k, (sp, color, curve) in enumerate((('mouse', MOUSE_C, M71['reference'][j]), ('human', HUMAN_C, M71['case'][j]))):
            whole = [np.log(y[P['specimen'] == n].sum() / P['library'][P['specimen'] == n].sum()) + LOG10K for n in PAIRS[sp]]
            ax_w.plot([k - .25, k + .25], [np.mean(whole)] * 2, color=color, lw=2)
            for n, val in zip(PAIRS[sp], whole):
                ax_w.scatter(k, val, color=color, marker=MARKER[n], s=12, zorder=3, edgecolor='white', linewidth=.4)
            for s_i, seg in enumerate(SEGMENTS):
                vals = []
                for n in PAIRS[sp]:
                    m = (P['specimen'] == n) & (P['segment'] == seg)
                    vals.append(np.log(max(y[m].sum(), .5) / P['library'][m].sum()) + LOG10K)
                xpos = s_i + (k - .5) * .38
                ax_s.plot([xpos - .15, xpos + .15], [np.mean(vals)] * 2, color=color, lw=2)
                for n, val in zip(PAIRS[sp], vals):
                    ax_s.scatter(xpos, val, color=color, marker=MARKER[n], s=12, zorder=3, edgecolor='white', linewidth=.4)
            points = []
            for n in PAIRS[sp]:
                m = P['specimen'] == n
                tot = pd.Series(y[m]).groupby(bins12[m]).sum()
                lib = pd.Series(P['library'][m]).groupby(bins12[m]).sum()
                centres = (tot.index.to_numpy() + .5) / EXAMPLE_BINS
                vals = np.log(np.maximum(tot.to_numpy(), .5) / lib.to_numpy()) + LOG10K
                points.extend(vals)
                ax_c.scatter(centres, vals, color=color, marker=MARKER[n], s=7, alpha=.8, edgecolor='white', linewidth=.3, zorder=3)
            ax_c.plot(P['grid'], curve + LOG10K, color=color, lw=1.3)
            limits = limits_c.get(r, []) + list(points)
            limits_c[r] = limits
        lo_c, hi_c = np.nanmin(limits_c[r]), np.nanmax(limits_c[r])
        pad = .15 * (hi_c - lo_c + 1e-9)
        ax_c.set_ylim(lo_c - pad, hi_c + pad)       # the curve is clipped where a species is nearly absent
        for t in (np.mean(DESC['P']['t12']), np.mean(DESC['P']['t23'])):
            ax_c.axvline(t, color=MUTED, lw=.6, ls=':')
        ax_w.set_xticks([0, 1], ['mouse', 'human'])
        ax_s.set_xticks([0, 1, 2], ['S1', 'S2', 'S3'])
        ax_w.set_ylabel(f'{g}\nln CP10k', fontsize=7)
        ax_w.set_title(f'{rule.split(" · ")[0]} whole PT', loc='left', fontsize=7)
        ax_s.set_title('S1 / S2 / S3', loc='left', fontsize=7)
        ax_c.set_title(f'{names[p][:58]}', loc='left', fontsize=7)
        for a in (ax_w, ax_s, ax_c):
            a.tick_params(labelsize=6)
            for side in ('top', 'right'):
                a.spines[side].set_visible(False)
    axes[-1, 2].set_xlabel('PT position (package pseudospace); dotted: mean S1→S2 and S2→S3 transitions', fontsize=7)
    axes[0, 2].legend(handles=[plt.Line2D([], [], color=MOUSE_C, lw=1.3, label='mouse'),
                               plt.Line2D([], [], color=HUMAN_C, lw=1.3, label='human'),
                               plt.Line2D([], [], color='#0b0b0b', marker='o', lw=0, markersize=3, label='specimen 1'),
                               plt.Line2D([], [], color='#0b0b0b', marker='s', lw=0, markersize=3, label='specimen 2')],
                      fontsize=5.5, frameon=False, ncol=4, loc='lower left', bbox_to_anchor=(0, 1.12))
    fig.tight_layout()
    for ext in ('pdf', 'png'):
        fig.savefig(figures / f'fig3a_worked_examples.{ext}', bbox_inches='tight', dpi=300)
    plt.show()

### R4 (iii) · Species shifts of a switch relative to each species' own boundaries

58's rule: monotone with one direction in all four specimens, sign agreement in all four cross-species pairs, and
|shift| beyond the registration threshold: the 95th percentile, over notebook 72's fold and half refits of the package
(failed refits ignored), of |Δ human − mouse gap| ÷ the human segment length. Programs are 75's. Claim if ≥ 10 genes
or ≥ 3 programs pass with the same sign on D.

In [ ]:
REG = pd.read_csv(nb72 / 'tables' / 'registration.csv')
REG = REG[REG.arm.eq('package') & REG.context.eq('hvm: all PT units')]
saved_gap = REG[REG.refit.eq('full')].set_index('segment').gap
refit_gaps = REG[REG.refit.isin(['fold0', 'fold1', 'fold2', 'half_A', 'half_B'])].dropna(subset=['gap'])
FOUR = (*MICE, *HUMANS)
SHIFT_COLUMNS = ['shift (segment units)', 'human onset segment', 'registration threshold', 'four-pair sign agreement',
                 'passes on P', 'shift on D', 'same sign on D', 'passes, same sign on D']
SHIFTS_EVALUABLE = all(f'{n}|ln|onset' in DESC['P'] for n in FOUR)
THRESHOLD = {}
if SHIFTS_EVALUABLE:
    human_lengths = np.mean([[DESC['P']['t12'][i] - P['grid'][0], DESC['P']['t23'][i] - DESC['P']['t12'][i],
                              P['grid'][-1] - DESC['P']['t23'][i]]
                             for i, n in enumerate(DESC['P']['specimens']) if n in HUMANS], axis=0)
    THRESHOLD = {k: float(((refit_gaps[refit_gaps.segment.eq(seg)].set_index('refit').gap - saved_gap[seg]).abs()
                           / human_lengths[k]).quantile(.95)) for k, seg in enumerate(SEGMENTS)}


def shifts_on(code):
    """58's (iii) inputs on one coordinate, or None when a specimen has no onsets there."""
    if not all(f'{n}|ln|onset' in DESC[code] for n in FOUR):
        return None
    onset = pd.DataFrame({n: DESC[code][f'{n}|ln|onset'] for n in FOUR}, index=GENES)
    direction = pd.DataFrame({n: DESC[code][f'{n}|ln|direction'] for n in FOUR}, index=GENES)
    mono = pd.DataFrame({n: DESC[code][f'{n}|ln|monotone'].astype(bool) for n in FOUR}, index=GENES)
    eligible_ = mono.all(axis=1) & direction.nunique(axis=1).eq(1)
    signs = pd.concat([np.sign(onset[h] - onset[m]) for h in HUMANS for m in MICE], axis=1)
    consistent = signs.nunique(axis=1).eq(1) & signs.iloc[:, 0].ne(0)
    shift = onset[list(HUMANS)].mean(axis=1) - onset[list(MICE)].mean(axis=1)
    seg = np.floor(onset[list(HUMANS)].mean(axis=1).clip(0, 2.999))
    return eligible_, consistent, shift, seg


SHIFTS = pd.DataFrame(columns=SHIFT_COLUMNS)
if SHIFTS_EVALUABLE:
    eligible_p, consistent_p, shift_p, seg_p = shifts_on('P')
    on_d = shifts_on('D')
    shift_d = on_d[2] if on_d is not None else pd.Series(np.nan, index=GENES)
    threshold_p = seg_p.map(THRESHOLD)
    passes_p = eligible_p & consistent_p & (shift_p.abs() > threshold_p)
    SHIFTS = pd.DataFrame({'shift (segment units)': shift_p, 'human onset segment': seg_p, 'registration threshold': threshold_p,
                           'four-pair sign agreement': consistent_p, 'passes on P': passes_p,
                           'shift on D': shift_d, 'same sign on D': np.sign(shift_d) == np.sign(shift_p)})[eligible_p]
    SHIFTS['passes, same sign on D'] = SHIFTS['passes on P'] & SHIFTS['same sign on D']
else:
    READING.append(f'R4 (iii) is not evaluable on P: {SKIPPED_P} has no ordered transitions, so no onsets in its own '
                   'segment units; no gene or program can pass, and the claim is not reached.')
SHIFTS.to_csv(tables / 'shifts_iii.csv')
PROGRAM_SHIFTS = pd.DataFrame()
if READY_75 and (T75 / 'programs_human_vs_mouse.csv').is_file():
    programs = pd.read_csv(T75 / 'programs_human_vs_mouse.csv')
    top10 = set(pd.Index(GENES)[STAT71['z'].to_numpy(float) >= np.quantile(STAT71['z'].to_numpy(float), .9)])
    rows_p = []
    for prog, members in programs.groupby('program'):
        if 'drivers' in members:
            drivers = sorted({g.strip() for x in members.drivers.dropna() for g in str(x).split(',')})
        else:
            genes_p = sorted(set().union(*[set(SETS[p]) for p in members.pathway_id if p in SETS]))
            drivers = [g for g in genes_p if g in top10] or genes_p
            drivers = drivers if len(drivers) >= PROGRAM_MIN_GENES else genes_p
        shifted = [g for g in drivers if g in SHIFTS.index and bool(SHIFTS.loc[g, 'passes, same sign on D'])]
        signs = {np.sign(SHIFTS.loc[g, 'shift (segment units)']) for g in shifted}
        rows_p.append({'program': prog, 'drivers': len(drivers), 'shifted drivers': len(shifted), 'one sign': len(signs) == 1,
                       'passes': len(shifted) >= PROGRAM_MIN_GENES and len(signs) == 1, 'genes': ', '.join(shifted)})
    PROGRAM_SHIFTS = pd.DataFrame(rows_p)
    PROGRAM_SHIFTS.to_csv(tables / 'shifts_iii_programs.csv', index=False)
    if 'drivers' not in programs:
        READING.append("R4 (iii): 75's programs carry no driver column; a program's drivers are its pathways' members in the "
                       "top 10% of 71's condition z (all members when fewer than three), 61's contributing rule.")
else:
    PENDING.append("R4 (iii) programs: 75's programs_human_vs_mouse.csv is absent (the gene count is reported)")
n_programs = int(PROGRAM_SHIFTS.passes.sum()) if len(PROGRAM_SHIFTS) else 0
SHIFT_CLAIM = bool(SHIFTS['passes, same sign on D'].astype(bool).sum() >= SHIFT_GENES or n_programs >= SHIFT_PROGRAMS)
print('Registration thresholds (segment units):', {SEGMENTS[k]: round(v, 3) for k, v in THRESHOLD.items()})
print(f"(iii) evaluable: {SHIFTS_EVALUABLE}; {len(SHIFTS)} genes monotone in all four specimens; "
      f"{int(SHIFTS['passes on P'].astype(bool).sum())} pass on P; {int(SHIFTS['passes, same sign on D'].astype(bool).sum())} "
      f"with the same sign on D; programs passing: {n_programs}; claim: {SHIFT_CLAIM}")
if len(SHIFTS):
    display(SHIFTS[SHIFTS['passes on P'].astype(bool)].sort_values('shift (segment units)').round(2))

## 12 · Sections that read notebook 72's tables (G4): the noise gate and the claims

- **C14** (72's rule, recomputed from `noise_budget.csv`): a cell (specimen × segment, package arm, all PT units) is
  noise-dominated when fold noise ÷ within-segment spread ≥ 1; C14 is not met when ≥ 6 of the 12 cells are. If C14 is
  not met, every within-segment result is descriptive and no within-segment claim is made.
- **Per-cell gate (new, O3):** a species × segment cell is noise-dominated when the mean fold noise ÷ spread of the
  species' two specimens is ≥ 1.

| Claim per species × segment | Requires |
|---|---|
| replicated within-segment gradients | C14 met; cell not noise-dominated; (i) ρ ≥ 0.3 and count-split ρ ≥ 0.2; ρ above every step-null replicate; ρ above the placebo's ρ |
| within-segment information in held-out genes | C14 met; cell not noise-dominated; WSR p ≤ 0.01 in both directions; above the placebo in both directions; the placebo itself not significant |

In [ ]:
READY_72 = all((nb72 / 'tables' / f).is_file() for f in ('noise_budget.csv', 'criteria_hvm.csv'))
C14_MET, N_DOMINATED, CELL_NOISE, CLAIMS = None, None, {}, None
NOISE = pd.DataFrame()
if READY_72:
    NOISE = pd.read_csv(nb72 / 'tables' / 'noise_budget.csv')
    criteria72 = pd.read_csv(nb72 / 'tables' / 'criteria_hvm.csv').set_index('criterion')
    noise_pkg = NOISE[NOISE.arm.eq('package') & NOISE.context.eq('hvm: all PT units')]
    assert len(noise_pkg) == 12, len(noise_pkg)
    N_DOMINATED = int((noise_pkg['fold noise ÷ spread'] >= NOISE_DOMINATED).sum())
    C14_MET = N_DOMINATED < C14_CELLS
    assert str(criteria72.loc['C14', 'value']).startswith(f'{N_DOMINATED} of 12'), criteria72.loc['C14', 'value']
    assert (criteria72.loc['C14', 'verdict'] == 'met') == C14_MET, 'G4: 72\'s C14 verdict differs'
    for sp, pair in PAIRS.items():
        for seg in SEGMENTS:
            cells = noise_pkg[noise_pkg.specimen.isin(pair) & noise_pkg.segment.eq(seg)]
            CELL_NOISE[(sp, SHORT_SEG[seg])] = float(cells['fold noise ÷ spread'].mean())
    claim_rows = []
    wsr = WSR.set_index(['arm', 'species', 'segment', 'train'])
    for sp, pair in PAIRS.items():
        for seg in ('S1', 'S2', 'S3'):
            key = f'(i) {sp} {seg} slope ρ'
            rho = OBSERVED[('P', 'ln')][key]
            null_max = COMPARISON[('P', 'ln')].loc[key, ['arm A max', 'arm B max']].max()
            split = split_rho.get(sp, {}).get(seg, np.nan)
            placebo = PL_SLOPES[sp][seg]
            gate = CELL_NOISE[(sp, seg)] < NOISE_DOMINATED
            p_sig = all(wsr.loc[('P', sp, seg, t), 'p'] <= WSR_ALPHA for t in pair)
            beats = all(wsr.loc[('P', sp, seg, t), 'WSR'] > wsr.loc[('PL', sp, seg, t), 'WSR'] for t in pair)
            placebo_sig = all(wsr.loc[('PL', sp, seg, t), 'p'] <= WSR_ALPHA for t in pair)
            parts = {'ρ ≥ 0.3': rho >= SLOPE_CLAIM, 'count-split ρ ≥ 0.2': split >= SLOPE_SPLIT,
                     'above every step null': rho > null_max, 'above the placebo': rho > placebo}
            wparts = {'WSR p ≤ 0.01 both directions': p_sig, 'WSR above the placebo both directions': beats,
                      'placebo WSR not significant': not placebo_sig}
            claim_rows.append({'species': sp, 'segment': seg, 'C14 met': C14_MET,
                               'mean fold noise ÷ spread': CELL_NOISE[(sp, seg)], 'cell not noise-dominated': gate,
                               'slope ρ (P)': rho, 'count-split ρ': split, 'step-null max': null_max, 'placebo ρ': placebo,
                               **parts, 'replicated within-segment gradients': bool(C14_MET and gate and all(parts.values())),
                               'gradients: all but C14 and the noise gate': bool(all(parts.values())),
                               'gradients: all but the placebo gate': bool(all(v for k, v in parts.items() if k != 'above the placebo')),
                               'WSR (P), mean of directions': float(np.mean([wsr.loc[('P', sp, seg, t), 'WSR'] for t in pair])),
                               'WSR (placebo), mean of directions': float(np.mean([wsr.loc[('PL', sp, seg, t), 'WSR'] for t in pair])),
                               **wparts, 'within-segment information in held-out genes': bool(C14_MET and gate and all(wparts.values())),
                               'held-out: all but C14 and the noise gate': bool(all(wparts.values()))})
    CLAIMS = pd.DataFrame(claim_rows)
    CLAIMS.to_csv(tables / 'cell_claims.csv', index=False)
    display(CLAIMS.T)
    print(f'C14: {N_DOMINATED} of 12 cells noise-dominated -> {"met" if C14_MET else "NOT met: every within-segment result is descriptive"}')
else:
    PENDING.append("R6 claims, R7 gate, Fig 3b-3d noise: notebook 72's noise_budget.csv or criteria_hvm.csv is absent")

### R7 · The transporter gate and Figure 3b

Figure 3b is shown as "micro-gradients" only if all of these hold: C14 met; W2 and W3 calibrated; ≥ 10 micro-gradient
transporters in at least one species; the T-restricted slope agreement (T ∩ fold-protected, > 20 genes) beyond every
step-null replicate in ≥ 1 non-noise-dominated cell. Otherwise it is a segment-level zonation heatmap without
within-segment marks. Every number is given for T and for T without SLC25 (O5).

In [ ]:
T_AGREE_ROWS = []
for label_t, genes_t in (('T', T_GENES), ('T without SLC25', T_NO25)):
    keep_t = FOLD_PROTECTED['P'] & pd.Index(GENES).isin(genes_t)
    for sp, pair in PAIRS.items():
        observed_t = slope_agreement(DESC['P'], pair, keep_t, min_genes=T_MIN_SLOPE_GENES)
        nulls_t = [slope_agreement(NULLS[('P', arm, rep)], pair, keep_t, min_genes=T_MIN_SLOPE_GENES)
                   for arm in 'AB' for rep in range(1, N_REPLICATES + 1)]
        for seg in ('S1', 'S2', 'S3'):
            null_max = np.nanmax([n[seg] for n in nulls_t]) if any(np.isfinite(n[seg]) for n in nulls_t) else np.nan
            T_AGREE_ROWS.append({'set': label_t, 'species': sp, 'segment': seg, 'genes': int(keep_t.sum()),
                                 'slope ρ': observed_t[seg], 'step-null max': null_max,
                                 'beyond every step null': bool(observed_t[seg] > null_max),
                                 'cell not noise-dominated': CELL_NOISE.get((sp, seg), np.nan) < NOISE_DOMINATED if CELL_NOISE else np.nan})
T_AGREE = pd.DataFrame(T_AGREE_ROWS)
camera_t = pd.concat([camera(W[(sp, 'W3')].z.dropna(), {'T': [g for g in T_GENES if g in W[(sp, 'W3')].z.dropna().index],
                                                         'T without SLC25': [g for g in T_NO25 if g in W[(sp, 'W3')].z.dropna().index]})
                      .assign(species=sp) for sp in PAIRS])
SUMMARY_T = []
for label_t, genes_t in (('T', T_GENES), ('T without SLC25', T_NO25)):
    sub = T_TABLE.loc[genes_t]
    agree = T_AGREE[T_AGREE.set.eq(label_t)]
    micro = {sp: int(sub[f'micro-gradient {sp}'].sum()) for sp in PAIRS}
    cell_ok = bool((agree['beyond every step null'] & agree['cell not noise-dominated'].fillna(False).astype(bool)).any())
    gate = {'C14 met': bool(C14_MET), 'W2 and W3 calibrated': W_CALIBRATED,
            '≥ 10 micro-gradient transporters in a species': max(micro.values()) >= T_MIN_CALLS,
            'slope agreement beyond the step nulls in a non-noise-dominated cell': cell_ok}
    SUMMARY_T.append({'set': label_t, 'genes': len(genes_t), **{f'micro-gradient genes, {sp}': micro[sp] for sp in PAIRS},
                      **{f'zonated (W1), {sp}': int(sub[f'zonated (W1 q ≤ 0.05) {sp}'].sum()) for sp in PAIRS},
                      **{f'cameraPR on W3 z, {sp}': '{} p {:.3g}'.format(*camera_t[camera_t.species.eq(sp) & camera_t.pathway_id.eq(label_t)]
                                                                          [['direction', 'p']].iloc[0]) for sp in PAIRS},
                      **gate, 'Figure 3b shows micro-gradients': all(gate.values())})
SUMMARY_T = pd.DataFrame(SUMMARY_T).set_index('set')
SUMMARY_T.to_csv(tables / 'transporter_summary.csv')
T_AGREE.to_csv(tables / 'transporter_slope_agreement.csv', index=False)
T_GATE = bool(SUMMARY_T.loc['T', 'Figure 3b shows micro-gradients'])
display(SUMMARY_T.T)
display(T_AGREE.round(3))
print('Micro-gradient genes among all universe genes:', ALL_MICRO)

# Figure 3b: transporters zonated in either species (W1 q <= 0.05), at most 60 by the larger W1 z
zon = T_TABLE[T_TABLE[[f'zonated (W1 q ≤ 0.05) {sp}' for sp in PAIRS]].any(axis=1)].copy()
zon['w1 max'] = zon[[f'W1 z {sp}' for sp in PAIRS]].max(axis=1)
zon = zon.sort_values('w1 max', ascending=False).head(60)
if len(zon):
    rows_h = [GENES.index(g) for g in zon.index]
    blocks_h = {'mouse': M71['reference'][rows_h], 'human': M71['case'][rows_h]}
    zs = {sp: (b - b.mean(axis=1, keepdims=True)) / np.maximum(b.std(axis=1, keepdims=True), 1e-9) for sp, b in blocks_h.items()}
    order = np.lexsort((zs['human'].argmax(axis=1), zs['mouse'].argmax(axis=1)))
    genes_h = list(zon.index[order])
    strips = {n: M71[f'curve|{n}'][rows_h] for n in (*MICE, *HUMANS)}
    zstrip = {}
    for sp, pair in PAIRS.items():
        mu, sd = blocks_h[sp].mean(axis=1, keepdims=True), np.maximum(blocks_h[sp].std(axis=1, keepdims=True), 1e-9)
        for n in pair:
            zstrip[n] = (strips[n] - mu) / sd                 # each specimen on its species' scale
    panels = [('mouse', zs['mouse']), (MICE[0], zstrip[MICE[0]]), (MICE[1], zstrip[MICE[1]]),
              ('human', zs['human']), (HUMANS[0], zstrip[HUMANS[0]]), (HUMANS[1], zstrip[HUMANS[1]])]
    fig, axes = plt.subplots(1, 8, figsize=(7.2, .12 * len(genes_h) + 1.6),
                             gridspec_kw={'width_ratios': [1, .45, .45, 1, .45, .45, .16, .5], 'wspace': .06})
    for ax, (title, values) in zip(axes[:6], panels):
        ax.imshow(values[order], aspect='auto', cmap='RdBu_r', vmin=-2.5, vmax=2.5,
                  extent=(P['grid'][0], P['grid'][-1], len(genes_h), 0), interpolation='nearest')
        names_here = PAIRS[title] if title in PAIRS else (title,)
        for i, n in enumerate(DESC['P']['specimens']):
            if n in names_here and (n not in SKIPPED_P):
                for t in (DESC['P']['t12'][i], DESC['P']['t23'][i]):
                    ax.axvline(t, color='#52514e', lw=.5, ls=':')
        ax.set_title(f'{title} (curve)' if title in PAIRS else title.replace('HUK1_', ''), fontsize=6, loc='left')
        ax.set_yticks(np.arange(len(genes_h)) + .5, genes_h if title == 'mouse' else [], fontsize=5)
        ax.set_xticks([.2, .5, .8])
        ax.tick_params(labelsize=5)
    axes[0].set_xlabel('PT position', fontsize=6)
    family_code = {'other solute carriers': 0, 'SLC25 (mitochondrial carriers)': 1, 'ABC transporters': 2}
    axes[6].imshow(np.array([[family_code[f]] for f in zon.family.reindex(genes_h)]), aspect='auto', cmap='Greys',
                   vmin=-1, vmax=3, extent=(0, 1, len(genes_h), 0))
    axes[6].set_xticks([]), axes[6].set_yticks([])
    axes[6].set_title('family', fontsize=6)
    marks = np.array([[T_TABLE.loc[g, f'micro-gradient {sp}'] and T_TABLE.loc[g, f'local gradient {sp} {seg}']
                       for sp in PAIRS for seg in ('S1', 'S2', 'S3')] for g in genes_h], float)
    axes[7].imshow(marks, aspect='auto', cmap='Greys', vmin=0, vmax=1.3, extent=(0, 6, len(genes_h), 0))
    for j, (sp, seg) in enumerate([(sp, seg) for sp in PAIRS for seg in ('S1', 'S2', 'S3')]):
        if CELL_NOISE and CELL_NOISE[(sp, seg)] >= NOISE_DOMINATED:
            axes[7].add_patch(plt.Rectangle((j, 0), 1, len(genes_h), fill=False, hatch='////', lw=0, color='#c3c2b7'))
    axes[7].set_xticks(np.arange(6) + .5, ['m S1', 'm S2', 'm S3', 'h S1', 'h S2', 'h S3'], fontsize=5, rotation=90)
    axes[7].set_yticks([])
    axes[7].set_title('within-segment\ncalls' if T_GATE else 'within-segment\n(descriptive)', fontsize=6)
    fig.text(.01, .005, 'Rows: transporters with a within-species positional gradient (W1 q ≤ 0.05), ordered by mouse then human '
             'peak; z within species (specimen strips on their species\' scale). Family: light = SLC, mid = SLC25 '
             '(mitochondrial), dark = ABC. Right: dark = micro-gradient gene (W2 and W3 q ≤ 0.05) with both specimens\' '
             'slopes of one sign (|·| ≥ 2) in that segment; hatched = noise-dominated cell (72).', fontsize=5, wrap=True)
    fig.suptitle('Figure 3b · solute transporters along the PT' + ('' if T_GATE else
                 ' — segment-level zonation; within-segment marks descriptive (gate not met)'), fontsize=7, x=.01, ha='left')
    for ext in ('pdf', 'png'):
        fig.savefig(figures / f'fig3b_transporters.{ext}', bbox_inches='tight', dpi=300)
    plt.show()
    zon.loc[genes_h].to_csv(tables / 'fig3b_rows.csv')

### Figures 3c and 3d: within-segment gradients against step nulls, and the resolution budget

**3c.** Per species × segment cell: the slope agreement on P and D (filled when beyond every step-null replicate), the
largest step-null replicate (dash), the placebo (grey cross) and the count split (open diamond).
**3d (60's layout).** Left: positional noise (fold noise ÷ within-segment spread, 72) against slope agreement, per
cell and coordinate; all PT units (P, package DPT) and, when O1 ran, the matched set M (every arm). Right: (iv) onset
order in mouse S2 against mouse S2 noise (fold SD in segment units). MC8 reads the Spearman over the M cells (all six
arms on one unit set), else over the all-unit cells.

In [ ]:
COLORS = {'P': '#2a78d6', 'D': '#eb6834', 'package': '#2a78d6', 'package DPT': '#eb6834', 'SCF13': '#1baf7a',
          'DPT13': '#9b59b6', 'LCP': '#eda100', 'LRS-Z': '#52514e'}
cells_x = [(sp, seg) for sp in PAIRS for seg in ('S1', 'S2', 'S3')]
fig, ax = plt.subplots(figsize=(4.6, 2.4))
for off, code in ((-.15, 'P'), (.15, 'D')):
    for j, (sp, seg) in enumerate(cells_x):
        key = f'(i) {sp} {seg} slope ρ'
        cmp = COMPARISON[(code, 'ln')].loc[key]
        ax.scatter(j + off, cmp.observed, s=30, color=COLORS[code], facecolor=COLORS[code] if cmp['exceeds every null replicate'] else 'white',
                   zorder=3, label=code if j == 0 else None)
        ax.plot([j + off - .08, j + off + .08], [max(cmp['arm A max'], cmp['arm B max'])] * 2, color='#52514e', lw=1.2)
for j, (sp, seg) in enumerate(cells_x):
    ax.scatter(j, PL_SLOPES[sp][seg], marker='x', color='#9a9a9a', s=20, zorder=3, label='placebo' if j == 0 else None)
    ax.scatter(j, split_rho.get(sp, {}).get(seg, np.nan), marker='D', facecolor='white', edgecolor='#0b0b0b', s=14, zorder=3,
               label='count split' if j == 0 else None)
ax.axhline(0, color='#c3c2b7', lw=.8)
ax.axhline(SLOPE_CLAIM, color='#c3c2b7', lw=.8, ls='--')
ax.set_xticks(range(6), [f'{sp[0]} {seg}' for sp, seg in cells_x], fontsize=6)
ax.set_ylabel('within-segment slopes:\nagreement between specimens', fontsize=6)
ax.tick_params(labelsize=6)
ax.legend(fontsize=5, frameon=False, ncol=4, loc='upper left')
ax.set_title('Figure 3c · filled = beyond every step null; dash = largest step null' +
             ('' if C14_MET else ' (C14 not met: descriptive)'), fontsize=6.5, loc='left')
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
for ext in ('pdf', 'png'):
    fig.savefig(figures / f'fig3c_within_segment_gradients.{ext}', bbox_inches='tight', dpi=300)
plt.show()

BUDGET_ROWS = []
if READY_72:
    def info(context, arm):
        if context == 'hvm: all PT units':
            code = {'package': 'P', 'package DPT': 'D'}[arm]
            return OBSERVED[(code, 'ln')], COMPARISON[(code, 'ln')]
        return M_OBSERVED.get(arm), M_COMPARISON.get(arm)

    arms_ctx = [('hvm: all PT units', 'package'), ('hvm: all PT units', 'package DPT')]
    arms_ctx += [('M · package labels', arm) for arm in M_ARMS]
    for context, arm in arms_ctx:
        obs_i, cmp_i = info(context, arm)
        if obs_i is None:
            continue
        nz = NOISE[NOISE.arm.eq(arm) & NOISE.context.eq(context)]
        for sp, pair in PAIRS.items():
            for seg in ('S1', 'S2', 'S3'):
                cell = nz[nz.specimen.isin(pair) & nz.segment.eq(f'PT-{seg}')]
                key = f'(i) {sp} {seg} slope ρ'
                BUDGET_ROWS.append({'context': context, 'arm': arm, 'species': sp, 'segment': seg,
                                    'fold noise ÷ spread': float(cell['fold noise ÷ spread'].mean()),
                                    'fold noise SD (segment units)': float(cell['fold noise SD (segment units)'].mean()),
                                    'count split noise ÷ spread': float(cell['count split noise ÷ spread'].mean())
                                    if 'count split noise ÷ spread' in cell else np.nan,
                                    'slope ρ': obs_i[key], 'step-null max': cmp_i.loc[key, ['arm A max', 'arm B max']].max(),
                                    'beyond step nulls': bool(cmp_i.loc[key, 'exceeds every null replicate']),
                                    '(iv) mouse S2 partial ρ': obs_i['(iv) mouse S2 partial ρ'],
                                    '(iv) 95% low': obs_i.get('(iv) 95% low', np.nan), '(iv) 95% high': obs_i.get('(iv) 95% high', np.nan),
                                    '(iv) step-null max': cmp_i.loc['(iv) mouse S2 partial ρ', ['arm A max', 'arm B max']].max(),
                                    '(iv) genes': obs_i['(iv) mouse S2 genes']})
BUDGET = pd.DataFrame(BUDGET_ROWS)
MC8_RHO, MC8_CONTEXT = np.nan, None
if len(BUDGET):
    BUDGET.to_csv(tables / 'resolution_budget.csv', index=False)
    use = BUDGET[BUDGET.context.eq('M · package labels')]
    MC8_CONTEXT = 'M · package labels' if use.arm.nunique() >= 3 else 'hvm: all PT units'
    use = BUDGET[BUDGET.context.eq(MC8_CONTEXT)].dropna(subset=['fold noise ÷ spread', 'slope ρ'])
    MC8_RHO = float(spearmanr(use['fold noise ÷ spread'], use['slope ρ']).statistic)
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.2, 2.8), gridspec_kw={'width_ratios': [1.5, 1]})
    for (context, arm), sub in BUDGET.groupby(['context', 'arm']):
        marker = 'o' if context.startswith('hvm') else '^'
        for _, r in sub.iterrows():
            ax.scatter(r['fold noise ÷ spread'], r['slope ρ'], s=28, marker=marker, color=COLORS.get(arm, '#0b0b0b'),
                       facecolor=COLORS.get(arm, '#0b0b0b') if r['beyond step nulls'] else 'white', linewidths=1.2, zorder=3)
        ax.scatter([], [], marker=marker, color=COLORS.get(arm, '#0b0b0b'),
                   label=f"{arm} ({'all units' if context.startswith('hvm') else 'M'})")
    ax.axvline(1, color='#c3c2b7', lw=.8, ls='--')
    ax.axhline(0, color='#c3c2b7', lw=.8)
    ax.set_xscale('log')
    ax.set_xticks([.1, .2, .5, 1, 2, 5], ['0.1', '0.2', '0.5', '1', '2', '5'])
    ax.xaxis.set_minor_formatter(plt.NullFormatter())
    ax.set_xlabel('positional noise ÷ within-segment spread (gene folds, notebook 72)', fontsize=6)
    ax.set_ylabel('within-segment slope agreement', fontsize=6)
    ax.legend(fontsize=5, frameon=False, ncol=4, loc='upper left', bbox_to_anchor=(0, -.22))
    ax.set_title(f'Filled = beyond every step null · Spearman over {MC8_CONTEXT} cells {MC8_RHO:.2f}', fontsize=6.5, loc='left')
    s2 = BUDGET[BUDGET.species.eq('mouse') & BUDGET.segment.eq('S2')]
    for _, r in s2.iterrows():
        x, color = r['fold noise SD (segment units)'], COLORS.get(r['arm'], '#0b0b0b')
        bx.plot([x, x], [r['(iv) 95% low'], r['(iv) 95% high']], color=color, lw=1.2)
        bx.scatter(x, r['(iv) mouse S2 partial ρ'], s=28, color=color, marker='o' if r['context'].startswith('hvm') else '^',
                   zorder=3)
        bx.scatter(x, r['(iv) step-null max'], marker='_', s=40, color='#52514e', zorder=3)
    bx.axhline(0, color='#c3c2b7', lw=.8)
    bx.set_xlabel('mouse S2 fold noise SD (segment units)', fontsize=6)
    bx.set_ylabel('(iv) onset order beyond r', fontsize=6)
    for a in (ax, bx):
        a.tick_params(labelsize=6)
        for side in ('top', 'right'):
            a.spines[side].set_visible(False)
    fig.suptitle('Figure 3d · resolution budget' + ('' if C14_MET else ' (C14 not met: within-segment information not claimed)'),
                 fontsize=7, x=.01, ha='left')
    fig.tight_layout()
    for ext in ('pdf', 'png'):
        fig.savefig(figures / f'fig3d_resolution_budget.{ext}', bbox_inches='tight', dpi=300)
    plt.show()
    display(BUDGET.round(3))

## 13 · Material change from the old record (protocol section 5) and the comparison with notebooks 57–61

Old and new differ in units, labels, coordinate and gene model, so every comparison is descriptive. A rule that is
met means the paper's text must change. MC1 and MC2 are judged on the primary (shared-step) design; 57's design is
reported beside them.

In [ ]:
p_dec = R1_DECISION.loc['P']
arm_b_med = max(p_dec['arm A median test B'], p_dec['arm B median test B'])
pass_b = R1_PASS[(R1_PASS.coordinate == 'P') & (R1_PASS.statistic == 'B')].iloc[0]['pass']
beyond_cells = [f'{sp} {seg}' for sp in PAIRS for seg in ('S1', 'S2', 'S3')
                if COMPARISON[('P', 'ln')].loc[f'(i) {sp} {seg} slope ρ', 'exceeds every null replicate']]
extrema_beyond = [f'{sp} {seg}' for sp in PAIRS for seg in ('S1', 'S2', 'S3')
                  if COMPARISON[('P', 'ln')].loc[f'(ii) {sp} {seg} replicated extrema', 'exceeds every null replicate']]
cross_cells = set(f'{r.species} {r.segment}' for r in cross[cross['counts (>= 2 of P, D, E including E)']].itertuples()) \
    if len(cross) else set()
claim_cells = [] if CLAIMS is None else [f'{a} {b}' for a, b, c in zip(CLAIMS.species, CLAIMS.segment,
                                                                          CLAIMS['replicated within-segment gradients']) if c]
lost_only = [] if CLAIMS is None else (
    [f'{a} {b} (gradients)' for a, b, core, claim in zip(CLAIMS.species, CLAIMS.segment,
                                                          CLAIMS['ρ ≥ 0.3'] & CLAIMS['count-split ρ ≥ 0.2'] & CLAIMS['above every step null'],
                                                          CLAIMS['replicated within-segment gradients']) if core and not claim]
    + [f'{a} {b} (held-out genes)' for a, b, core, claim in zip(CLAIMS.species, CLAIMS.segment,
                                                                CLAIMS['WSR p ≤ 0.01 both directions'],
                                                                CLAIMS['within-segment information in held-out genes']) if core and not claim])
mc3 = None
if CONT_VS_SEG is not None:
    clusters = CONT_VS_SEG.set_index('unit').loc['redundancy clusters']
    mc3 = bool(clusters['continuous only (A∖S)'] == 0 or clusters['segment only (S∖A)'] > clusters['continuous only (A∖S)']
               or S_PASS['pass'] == 'fail')
MC = pd.DataFrame([
    {'#': 'MC1', 'rule': 'R1 decision differs from "discordance"', 'old': '0.51 (discordance)',
     'new': f"{p_dec['primary ratio (arm A test A ÷ observed A)']:.2f} ({p_dec['decision']})",
     'met': not p_dec['decision'].startswith('label–coordinate discordance')},
    {'#': 'MC2', 'rule': 'test B calls ≤ 2 × the larger arm median, or test B fails the pass rule', 'old': '67 vs 9–11; pass',
     'new': f"{p_dec['test B calls (beyond coordinate steps)']} vs {arm_b_med:g}; {pass_b}",
     'met': bool(p_dec['test B calls (beyond coordinate steps)'] <= 2 * arm_b_med or pass_b == 'fail')},
    {'#': 'MC3', 'rule': 'A∖S is 0 clusters, or S∖A > A∖S in clusters, or the segment F-test fails its pass rule',
     'old': 'adds 13, steps add 3',
     'new': 'pending (notebook 75)' if CONT_VS_SEG is None else
     '{} / {} clusters; segment F-test {}'.format(clusters['continuous only (A∖S)'], clusters['segment only (S∖A)'], S_PASS['pass']),
     'met': mc3},
    {'#': 'MC4', 'rule': 'no cell meets the gradients claim, or a cell other than mouse S2 meets it', 'old': 'mouse S2',
     'new': 'pending (notebook 72)' if CLAIMS is None else (', '.join(claim_cells) or 'no cell'),
     'met': None if CLAIMS is None else bool(not claim_cells or any(c != 'mouse S2' for c in claim_cells))},
    {'#': 'MC5', 'rule': '(iv) becomes claim or abandon', 'old': '0.36 (0.16–0.52); inconclusive',
     'new': f"{iv_value:.2f} ({primary_iv['95% low']:.2f}–{primary_iv['95% high']:.2f}); {DECISION_IV}",
     'met': not DECISION_IV.startswith('inconclusive')},
    {'#': 'MC6', 'rule': '(ii): a mouse cell beyond the step nulls on P with cross-coordinate genes (P, D, E incl. E), or human S1 '
     'not beyond the nulls', 'old': 'human S1 only', 'new': f"beyond nulls on P: {', '.join(extrema_beyond) or 'none'}"
     + (f"; human pair not evaluable ({', '.join(SKIPPED_P)} has no ordered transitions)" if any(n in HUMANS for n in SKIPPED_P) else ''),
     'met': bool(any(c.startswith('mouse') and c in cross_cells for c in extrema_beyond) or 'human S1' not in extrema_beyond)},
    {'#': 'MC7', 'rule': '(iii) or R5 reaches its claim', 'old': '5 genes; external 0.10',
     'new': ("(iii) not evaluable" if not SHIFTS_EVALUABLE else f"{int(SHIFTS['passes, same sign on D'].astype(bool).sum())} genes")
            + f" (claim {SHIFT_CLAIM}); external {ext_value:.2f} ({DECISION_R5})",
     'met': bool(SHIFT_CLAIM or DECISION_R5.startswith('claim'))},
    {'#': 'MC8', 'rule': 'noise vs slope-agreement Spearman over cells ≥ 0', 'old': '−0.31 (24 cells)',
     'new': 'pending (notebook 72)' if MC8_CONTEXT is None else f'{MC8_RHO:.2f} ({MC8_CONTEXT})',
     'met': None if MC8_CONTEXT is None else bool(MC8_RHO >= 0)},
    {'#': 'MC9', 'rule': 'a claim is lost only through the new gates (C14, the per-cell noise gate, the placebo)',
     'old': '— (the gates are new)',
     'new': 'pending (notebook 72)' if CLAIMS is None else (', '.join(lost_only) or 'none'),
     'met': None if CLAIMS is None else bool(lost_only)}])
MC.to_csv(tables / 'material_change.csv', index=False)
display(MC)

def ladder_value(ds, row):
    r = LADDER[(LADDER.dataset == DS_NAME[ds]) & (LADDER.row == row)].iloc[0]
    return f"{r.calls} ({r['relabeled calls']}); {r['pass']}"


COMPARE_OLD = pd.DataFrame([
    {'result': 'R1 test A calls (relabeled)', 'old (SCF13, Gaussian)': '79 (6, 0)',
     'new (package, NB)': f"{p_dec['test A calls (beyond label steps)']} ({p_dec['relabeled test A']})"},
    {'result': 'R1 arm A median test A; ratio', 'old (SCF13, Gaussian)': '40; 0.51',
     'new (package, NB)': f"{p_dec['arm A median test A (range)']}; {p_dec['primary ratio (arm A test A ÷ observed A)']:.2f}"},
    {'result': 'R1 test B calls (relabeled); arm medians', 'old (SCF13, Gaussian)': '67 (1, 0); 9–11',
     'new (package, NB)': f"{p_dec['test B calls (beyond coordinate steps)']} ({p_dec['relabeled test B']}); "
                          f"{p_dec['arm A median test B']:g}, {p_dec['arm B median test B']:g}"},
    {'result': 'R1 floor (arm B test A, arm A test B)', 'old (SCF13, Gaussian)': '3–31', 'new (package, NB)': p_dec['floor: arm B test A, arm A test B (medians)']},
    {'result': "R1 57's design (tests A, B)", 'old (SCF13, Gaussian)': '79, 67', 'new (package, NB)': p_dec["57's design: test A, test B calls"]},
    {'result': 'R1 within-segment candidates (strict)', 'old (SCF13, Gaussian)': '48 (20 strict on SCF13)',
     'new (package, NB)': f"{int(CAND['within-segment candidate (P and D)'].sum())} ({int(CAND['strict (P set z above every simulated)'].sum())})"},
    {'result': 'R2 continuous adds / steps add (sets)', 'old (SCF13, Gaussian)': '13 / 3',
     'new (package, NB)': 'pending (75)' if CONT_VS_SEG is None else
     f"{CONT_VS_SEG.set_index('unit').loc['sets', 'continuous only (A∖S)']} / {CONT_VS_SEG.set_index('unit').loc['sets', 'segment only (S∖A)']}"},
    {'result': 'R2 steps on the spline, human vs mouse', 'old (SCF13, Gaussian)': '16 (0, 0)', 'new (package, NB)': ladder_value('hvm', 'steps_on_spline')},
    {'result': 'R2 six bins, human vs mouse', 'old (SCF13, Gaussian)': '49 (17, 0)', 'new (package, NB)': ladder_value('hvm', 'six_bins')},
    {'result': 'R4 (iv) mouse S2 partial ρ; decision', 'old (SCF13, Gaussian)': '0.36 (0.16–0.52), null max 0.49; inconclusive',
     'new (package, NB)': f"{iv_value:.2f} ({primary_iv['95% low']:.2f}–{primary_iv['95% high']:.2f}), null max "
                          f"{COMPARISON[('P', 'ln')].loc['(iv) mouse S2 partial ρ', ['arm A max', 'arm B max']].max():.2f}; {DECISION_IV}"},
    {'result': 'R4 (i) cells beyond every step null', 'old (SCF13, Gaussian)': 'mouse S2, human S1', 'new (package, NB)': ', '.join(beyond_cells) or 'none'},
    {'result': 'R4 (ii) cells beyond every step null', 'old (SCF13, Gaussian)': 'human S1', 'new (package, NB)': ', '.join(extrema_beyond) or 'none'},
    {'result': 'R4 (iii) genes passing (programs)', 'old (SCF13, Gaussian)': '5 (0)',
     'new (package, NB)': f"not evaluable ({', '.join(SKIPPED_P)} transitions out of order)" if not SHIFTS_EVALUABLE else
     f"{int(SHIFTS['passes, same sign on D'].astype(bool).sum())} ({int(PROGRAM_SHIFTS.passes.sum()) if len(PROGRAM_SHIFTS) else 0})"},
    {'result': 'R5 external onset partial ρ', 'old (SCF13, Gaussian)': '0.10 (0.01–0.24)', 'new (package, NB)': f'{ext_value:.2f} (low {ext_low:.2f})'},
    {'result': 'R6 fold noise ÷ spread (package, species means)', 'old (SCF13, Gaussian)': '0.83–1.49',
     'new (package, NB)': 'pending (72)' if not CELL_NOISE else f"{min(CELL_NOISE.values()):.2f}–{max(CELL_NOISE.values()):.2f}"},
    {'result': 'R6 noise vs slope agreement', 'old (SCF13, Gaussian)': '−0.31', 'new (package, NB)': 'pending (72)' if MC8_CONTEXT is None else f'{MC8_RHO:.2f}'}])
COMPARE_OLD.to_csv(tables / 'compare_old.csv', index=False)
display(COMPARE_OLD)

## 14 · How the protocol was read, pending sections, and the run record

In [ ]:
READING += [
    'R1: test B\'s coordinate-threshold segments are recomputed per coordinate (P, D); the arms\' permutation blocks use them.',
    'R1 strict candidates compare each observed set z on P with the largest simulated set z of the same test over both arms '
    '(57\'s rule on its primary coordinate).',
    'R4 nulls compute the (iv) partial correlation without its bootstrap (the estimate is the bootstrap\'s point value).',
    'R4 count split: (iv) and (i) use all genes (58\'s rule); a second (iv) row excludes the half-A refit\'s integration features.',
    'R4 E (equal depth) is run for the extrema rule and reported for (i) and (iv) without step nulls.',
    'M comparators: fold protection excludes each arm\'s construction genes and the 15 panel and anchor genes, which set the '
    'labels every arm uses on M; no (i-b) slopes on M.',
    'R5: the log1p sensitivity uses our own log1p onsets (R4) against the external log1p onsets.',
    'R7: within-species tests use the genes with counts in both specimens of that species (a gene without counts in a '
    'specimen has no specimen contrast); moderation and BH run over those genes.',
    'R7 slope agreement on T needs > 20 genes in a cell (58 needed > 50 over all genes).',
    'MC6: "a mouse cell exceeds the nulls on P and E" is read as a mouse cell beyond every step null on P that also holds '
    'genes meeting the cross-coordinate rule (P, D, E including E); E has no step nulls of its own.',
    'MC8 uses the M cells (all six arms on one unit set) when the comparator section ran, else the all-unit cells.',
    'Placebo refit: the package\'s default threads and file layout, as notebooks 70 and 72 ran it.']
print('How the protocol was read:')
print('\n'.join(f'{i + 1}. {line}' for i, line in enumerate(READING)))
print('\nPending:', PENDING or 'none')
RECORD = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'package': EXTERNAL, 'fingerprints': CONFIRMED_PASS1,
          'units': {code: int(len(v['ids'])) for code, v in V.items()}, 'genes': len(GENES), 'sets': len(SETS),
          'G5': G5, 'C14_met': C14_MET, 'noise_dominated_cells': N_DOMINATED,
          'r1_decision': R1_DECISION['decision'].to_dict(), 'iv_decision': DECISION_IV, 'r5_decision': DECISION_R5,
          'shift_claim': SHIFT_CLAIM, 'transporter_gate': T_GATE if READY_72 else None,
          'material_change': MC[['#', 'met']].astype(str).set_index('#')['met'].to_dict(),
          'reading': READING, 'pending': PENDING, 'stage_seconds': SECONDS,
          'wall_seconds': time.perf_counter() - NOTEBOOK_START, 'n_jobs': N_JOBS}
(output / 'run_record.json').write_text(json.dumps(RECORD, indent=2, default=str))
print(f"Wall time {RECORD['wall_seconds'] / 60:.1f} min")